# Credit Analytics Engine: FULL NOTEBOOK
**One file, every step.** You don't need any other file. All the code lives inside this notebook.

| Section | What it does |
|---|---|
| **A** | Setup (writes and loads the code) |
| **B** | Day 1: bond pricing, duration, DV01, CS01 + validation |
| **C** | Downloads and checks real FRED data |
| **D** | Spreads, data gaps, proxy validation, charts |
| **E** | Regime model: calm / stress (Markov Switching), full-sample (in-sample) |
| **F** | Live simulation: expanding-window periodic refit, min_train_weeks sensitivity check |
| **G** | Risk contribution: DV01=CS01, but multiplied by real volatility, which factor dominates |
| **H** | Stress scenarios: parallel shock, steepener, historical-style spread widening, portfolio P&L |
| **I** | Advanced interactive dashboard: pricing, scenario shocks and the live regime, in one panel |

## How to run it
1. (Recommended, 1 minute) **Put your FRED API key in Colab Secrets.** Never type the key into a cell.
   Get a **new key** at fred.stlouisfed.org > My Account > API Keys. In Colab, click the **key icon (Secrets)** in the left panel > *Add new secret*:
   Name `FRED_API_KEY`, Value = your key, **turn on Notebook access**.
2. From the menu: **Runtime > Run all**.
3. Scroll to the bottom once it's done.

**What to send back:** only the cells whose heading says "Send me this" (a screenshot is fine). Section B is for your own confidence: the numbers should match Day 1.
If something errors, paste the **whole** error, don't change anything yourself. (v0.8.0)

# SECTION A: Setup
## A1. Clean slate
Removes anything left over from earlier attempts.

In [ ]:
import os, sys, shutil, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "credit-engine"], check=False)
for m in [m for m in sys.modules if m.startswith("credit_engine")]:
    del sys.modules[m]                                   # drop any old version cached in memory
shutil.rmtree("credit_engine_src", ignore_errors=True)   # remove old files
for d in ["pricing", "risk", "data", "spreads", "portfolio"]:
    os.makedirs(f"credit_engine_src/credit_engine/{d}", exist_ok=True)
    open(f"credit_engine_src/credit_engine/{d}/__init__.py", "w").close()
print("Clean slate done.")

## A2. Write the source files
Each cell below creates one source file. Just run and move on; open one up if you're curious.

In [ ]:
%%writefile credit_engine_src/credit_engine/__init__.py
"""credit_engine: bond pricing, interest-rate risk and credit-spread risk."""
__version__ = "0.8.0"


In [ ]:
%%writefile credit_engine_src/credit_engine/pricing/daycount.py
"""Day-count conventions.

Only what a fixed-rate bullet bond needs: for a given coupon period we want
the fraction of the period that has *elapsed* (-> accrued interest) and the
fraction that *remains* (-> first exponent in the discounting formula).

Supported:
    30/360 US (bond basis)  - US corporate bonds
    ACT/ACT ICMA            - US Treasuries, Eurobonds
"""
from __future__ import annotations

import calendar
from datetime import date
from enum import Enum


class DayCount(str, Enum):
    THIRTY_360 = "30/360"
    ACT_ACT_ICMA = "ACT/ACT"


def _is_last_day_of_feb(d: date) -> bool:
    return d.month == 2 and d.day == calendar.monthrange(d.year, 2)[1]


def days_30_360_us(d1: date, d2: date) -> int:
    """30/360 US day count (ISDA/SIA 'bond basis' with the February rules).

    Rules, applied in this order:
      1. if both d1 and d2 are the last day of February -> D2 = 30
      2. if d1 is the last day of February              -> D1 = 30
      3. if D2 == 31 and D1 >= 30                       -> D2 = 30
      4. if D1 == 31                                    -> D1 = 30
    """
    dd1, dd2 = d1.day, d2.day
    if _is_last_day_of_feb(d1) and _is_last_day_of_feb(d2):
        dd2 = 30
    if _is_last_day_of_feb(d1):
        dd1 = 30
    if dd2 == 31 and dd1 >= 30:
        dd2 = 30
    if dd1 == 31:
        dd1 = 30
    return 360 * (d2.year - d1.year) + 30 * (d2.month - d1.month) + (dd2 - dd1)


def period_fractions(
    convention: DayCount,
    last_coupon: date,
    settlement: date,
    next_coupon: date,
    frequency: int,
) -> tuple[float, float]:
    """Return (elapsed, remaining) fractions of the current coupon period.

    elapsed   = share of the period between last coupon and settlement
    remaining = 1 - elapsed

    Why ``remaining = 1 - elapsed`` and not a second day count from settlement
    to the next coupon: with 30/360 the two direct counts can add up to 361 or
    359 days when a coupon date is the 31st or the end of February. The industry
    standard (SIA: DSC = E - A) and QuantLib both define the days to the next
    coupon as the period length minus the accrued days, so the two pieces always
    sum to exactly one period.
    """
    if convention is DayCount.THIRTY_360:
        elapsed = days_30_360_us(last_coupon, settlement) / (360.0 / frequency)
    elif convention is DayCount.ACT_ACT_ICMA:
        elapsed = (settlement - last_coupon).days / (next_coupon - last_coupon).days
    else:  # pragma: no cover
        raise ValueError(f"Unsupported day count: {convention}")
    return elapsed, 1.0 - elapsed


In [ ]:
%%writefile credit_engine_src/credit_engine/pricing/bond.py
"""Fixed-rate bullet bond: cash flows, accrued interest, clean/dirty price.

Conventions (documented on purpose, because most 'bugs' in bond code are
convention mismatches):

* Prices are quoted per 100 of par. Position size is handled by a separate
  ``notional`` argument in the risk functions.
* Yields are annualised, periodically compounded at the coupon frequency
  ("street convention"): DF(n) = (1 + y/f) ** (-n), n = coupon periods.
* Settlement on a coupon date: that coupon is already paid (accrued = 0) and
  the next coupon is one full period away.
* Coupon dates are generated backwards from maturity (so a 31 Aug maturity
  gives 28/29 Feb and 31 Aug coupons - no drift).
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date
from typing import Callable

import numpy as np
from dateutil.relativedelta import relativedelta

from .daycount import DayCount, period_fractions

ZeroCurve = Callable[[np.ndarray], np.ndarray]  # years -> zero rate (periodic comp.)


@dataclass(frozen=True)
class Cashflows:
    """Remaining cash flows seen from a settlement date (per 100 par)."""

    periods: np.ndarray   # time to each flow in coupon periods: w, w+1, w+2, ...
    amounts: np.ndarray   # coupon (and par at the end)
    accrued: float        # accrued interest per 100 par
    last_coupon: date
    next_coupon: date
    frequency: int

    @property
    def years(self) -> np.ndarray:
        return self.periods / self.frequency


@dataclass(frozen=True)
class Bond:
    coupon_rate: float                      # annual, decimal (0.05 = 5%)
    maturity: date
    frequency: int = 2                      # coupons per year
    day_count: DayCount = DayCount.THIRTY_360

    def __post_init__(self) -> None:
        if self.frequency not in (1, 2, 4, 12):
            raise ValueError("frequency must be 1, 2, 4 or 12")
        if self.coupon_rate < 0:
            raise ValueError("coupon_rate must be >= 0")

    # ------------------------------------------------------------------ dates
    def _coupon_dates(self, settlement: date) -> tuple[date, list[date]]:
        """Return (last coupon <= settlement, remaining coupon dates ascending)."""
        if settlement >= self.maturity:
            raise ValueError("settlement must be before maturity")
        step = 12 // self.frequency
        remaining: list[date] = []
        k = 0
        while True:
            d = self.maturity - relativedelta(months=step * k)
            if d <= settlement:
                last = d
                break
            remaining.append(d)
            k += 1
        remaining.reverse()
        return last, remaining

    # -------------------------------------------------------------- cashflows
    def cashflows(self, settlement: date) -> Cashflows:
        last, remaining = self._coupon_dates(settlement)
        nxt = remaining[0]
        elapsed, rem = period_fractions(
            self.day_count, last, settlement, nxt, self.frequency
        )
        coupon = 100.0 * self.coupon_rate / self.frequency
        n = len(remaining)
        amounts = np.full(n, coupon)
        amounts[-1] += 100.0
        periods = rem + np.arange(n, dtype=float)
        return Cashflows(
            periods=periods,
            amounts=amounts,
            accrued=coupon * elapsed,
            last_coupon=last,
            next_coupon=nxt,
            frequency=self.frequency,
        )

    def accrued_interest(self, settlement: date) -> float:
        return self.cashflows(settlement).accrued


# ------------------------------------------------------------------- pricing
def dirty_price(bond: Bond, settlement: date, ytm: float) -> float:
    """Full price (clean + accrued) per 100 par at a periodic-compounded yield."""
    cf = bond.cashflows(settlement)
    f = cf.frequency
    return float(np.sum(cf.amounts * (1.0 + ytm / f) ** (-cf.periods)))


def clean_price(bond: Bond, settlement: date, ytm: float) -> float:
    cf = bond.cashflows(settlement)
    f = cf.frequency
    dirty = float(np.sum(cf.amounts * (1.0 + ytm / f) ** (-cf.periods)))
    return dirty - cf.accrued


def dirty_price_from_curve(
    bond: Bond,
    settlement: date,
    zero_curve: ZeroCurve,
    spread: float = 0.0,
) -> float:
    """Discount each flow at zero(t) + spread (both periodic-compounded).

    This is what lets us separate two risk factors:
      * shifting ``zero_curve``  -> interest-rate risk (DV01)
      * shifting ``spread``      -> credit-spread risk (CS01)
    With a flat curve z = y - s this reproduces ``dirty_price(..., y)`` exactly.
    """
    cf = bond.cashflows(settlement)
    f = cf.frequency
    z = np.asarray(zero_curve(cf.years), dtype=float)
    return float(np.sum(cf.amounts * (1.0 + (z + spread) / f) ** (-cf.periods)))


In [ ]:
%%writefile credit_engine_src/credit_engine/pricing/curves.py
"""Minimal zero-curve helpers (periodic compounding at the bond's frequency).

Nelson-Siegel / Svensson fitting is planned for v0.2; for now a flat curve and
a piecewise-linear curve are enough to test the spread/rate risk split.
"""
from __future__ import annotations

import numpy as np

from .bond import ZeroCurve


def flat_curve(rate: float) -> ZeroCurve:
    return lambda t: np.full_like(np.asarray(t, dtype=float), rate)


def linear_curve(tenors: list[float], rates: list[float]) -> ZeroCurve:
    """Linear interpolation, flat extrapolation outside the given tenors."""
    x = np.asarray(tenors, dtype=float)
    y = np.asarray(rates, dtype=float)
    return lambda t: np.interp(np.asarray(t, dtype=float), x, y)


def shift_curve(curve: ZeroCurve, bump: float) -> ZeroCurve:
    """Parallel shift (bump in decimal, 1bp = 1e-4)."""
    return lambda t: np.asarray(curve(t), dtype=float) + bump


In [ ]:
%%writefile credit_engine_src/credit_engine/pricing/yield_solver.py
"""Yield-to-maturity solvers: Brent (robust) and Newton (fast, analytic slope)."""
from __future__ import annotations

from datetime import date
from typing import Literal

import numpy as np
from scipy.optimize import brentq

from .bond import Bond, dirty_price

PriceType = Literal["clean", "dirty"]

_Y_LO, _Y_HI = -0.5, 5.0   # search bracket for the yield (-50% .. 500%)


def _target_dirty(bond: Bond, settlement: date, price: float, price_type: PriceType) -> float:
    if price <= 0:
        raise ValueError("price must be positive")
    if price_type == "clean":
        return price + bond.accrued_interest(settlement)
    if price_type == "dirty":
        return price
    raise ValueError("price_type must be 'clean' or 'dirty'")


def ytm_brent(bond: Bond, settlement: date, price: float,
              price_type: PriceType = "clean", tol: float = 1e-14) -> float:
    """Yield via Brent's method. Price is strictly decreasing in y, so a sign
    change on the bracket guarantees a unique root."""
    target = _target_dirty(bond, settlement, price, price_type)
    g = lambda y: dirty_price(bond, settlement, y) - target
    lo, hi = _Y_LO, _Y_HI
    if g(lo) * g(hi) > 0:
        raise ValueError(f"price {price} is outside the solvable range for this bond")
    return float(brentq(g, lo, hi, xtol=tol, rtol=1e-14, maxiter=200))


def ytm_newton(bond: Bond, settlement: date, price: float,
               price_type: PriceType = "clean", y0: float | None = None,
               tol: float = 1e-12, max_iter: int = 50) -> float:
    """Yield via Newton-Raphson with the analytic dP/dy.

    Faster than Brent but can diverge for deeply distressed prices; keep Brent
    as the default and use this one as an independent cross-check.
    """
    target = _target_dirty(bond, settlement, price, price_type)
    cf = bond.cashflows(settlement)
    f = cf.frequency
    y = bond.coupon_rate if y0 is None else y0
    for _ in range(max_iter):
        base = 1.0 + y / f
        if base <= 0.05:
            raise RuntimeError("Newton left the valid domain; use ytm_brent")
        pv = np.sum(cf.amounts * base ** (-cf.periods))
        slope = np.sum(-cf.amounts * (cf.periods / f) * base ** (-cf.periods - 1.0))
        step = (pv - target) / slope
        y -= step
        if abs(step) < tol:
            return float(y)
    raise RuntimeError("Newton did not converge; use ytm_brent")


def solve_ytm(bond: Bond, settlement: date, price: float,
              price_type: PriceType = "clean", method: Literal["brent", "newton"] = "brent") -> float:
    if method == "brent":
        return ytm_brent(bond, settlement, price, price_type)
    if method == "newton":
        return ytm_newton(bond, settlement, price, price_type)
    raise ValueError("method must be 'brent' or 'newton'")


In [ ]:
%%writefile credit_engine_src/credit_engine/risk/duration.py
"""Interest-rate risk measures for a fixed-rate bond.

Notation (per 100 par, yield y compounded f times a year, n_i = periods to flow i):

    P      = sum_i  A_i (1+y/f)^(-n_i)                       dirty price
    D_mac  = sum_i (n_i/f) * PV_i / P                        Macaulay duration (years)
    D_mod  = D_mac / (1 + y/f)                               = -(1/P) dP/dy
    C      = (1/P) sum_i A_i n_i (n_i+1) / f^2 (1+y/f)^(-n_i-2)   = (1/P) d2P/dy2
    DV01   = D_mod * P * 1e-4                                price change for +1bp

Everything is measured on the *dirty* price (that is the actual money at risk).
Sign convention: DV01 is reported as a positive number = loss for a +1bp move.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import numpy as np

from ..pricing.bond import Bond, dirty_price

ONE_BP = 1e-4


def _pv_terms(bond: Bond, settlement: date, ytm: float):
    cf = bond.cashflows(settlement)
    f = cf.frequency
    base = 1.0 + ytm / f
    pv = cf.amounts * base ** (-cf.periods)
    return cf, base, pv


# ------------------------------------------------------------------ analytic
def macaulay_duration(bond: Bond, settlement: date, ytm: float) -> float:
    cf, _, pv = _pv_terms(bond, settlement, ytm)
    return float(np.sum(cf.years * pv) / np.sum(pv))


def modified_duration(bond: Bond, settlement: date, ytm: float) -> float:
    f = bond.frequency
    return macaulay_duration(bond, settlement, ytm) / (1.0 + ytm / f)


def convexity(bond: Bond, settlement: date, ytm: float) -> float:
    cf, base, pv = _pv_terms(bond, settlement, ytm)
    f = cf.frequency
    n = cf.periods
    second = np.sum(cf.amounts * n * (n + 1.0) / f**2 * base ** (-n - 2.0))
    return float(second / np.sum(pv))


def dv01(bond: Bond, settlement: date, ytm: float, notional: float = 100.0) -> float:
    """Money lost for a +1bp yield move on a position of ``notional`` par."""
    price = dirty_price(bond, settlement, ytm)
    position_value = price / 100.0 * notional
    return modified_duration(bond, settlement, ytm) * position_value * ONE_BP


# ------------------------------------------------------------ bump-and-reprice
def effective_duration(bond: Bond, settlement: date, ytm: float, bump: float = ONE_BP) -> float:
    """(P- - P+) / (2 P0 dy): finite-difference duration, no formula needed."""
    p0 = dirty_price(bond, settlement, ytm)
    up = dirty_price(bond, settlement, ytm + bump)
    dn = dirty_price(bond, settlement, ytm - bump)
    return (dn - up) / (2.0 * p0 * bump)


def effective_convexity(bond: Bond, settlement: date, ytm: float, bump: float = ONE_BP) -> float:
    """(P+ + P- - 2 P0) / (P0 dy^2)."""
    p0 = dirty_price(bond, settlement, ytm)
    up = dirty_price(bond, settlement, ytm + bump)
    dn = dirty_price(bond, settlement, ytm - bump)
    return (up + dn - 2.0 * p0) / (p0 * bump**2)


def price_change_estimate(mod_dur: float, conv: float, price: float, dy: float) -> float:
    """Second-order Taylor: dP ~ P * (-D_mod dy + 0.5 C dy^2)."""
    return price * (-mod_dur * dy + 0.5 * conv * dy**2)


# ------------------------------------------------------------------- summary
@dataclass(frozen=True)
class RateRisk:
    ytm: float
    clean_price: float
    dirty_price: float
    accrued: float
    macaulay: float
    modified: float
    convexity: float
    dv01: float
    effective_duration: float
    effective_convexity: float


def rate_risk_report(bond: Bond, settlement: date, ytm: float, notional: float = 100.0) -> RateRisk:
    dirty = dirty_price(bond, settlement, ytm)
    accrued = bond.accrued_interest(settlement)
    return RateRisk(
        ytm=ytm,
        clean_price=dirty - accrued,
        dirty_price=dirty,
        accrued=accrued,
        macaulay=macaulay_duration(bond, settlement, ytm),
        modified=modified_duration(bond, settlement, ytm),
        convexity=convexity(bond, settlement, ytm),
        dv01=dv01(bond, settlement, ytm, notional),
        effective_duration=effective_duration(bond, settlement, ytm),
        effective_convexity=effective_convexity(bond, settlement, ytm),
    )


In [ ]:
%%writefile credit_engine_src/credit_engine/risk/cs01.py
"""Credit-spread risk (CS01) and Z-spread, separated from rate risk.

Price = sum_i A_i * (1 + (z(t_i) + s)/f) ** (-n_i)
    z(t): risk-free zero curve      s: credit spread (Z-spread)

    DV01_curve : bump the whole curve z by 1bp, spread s held fixed
    CS01       : bump the spread s by 1bp, curve z held fixed

IMPORTANT (a finding worth writing in the README):
For a plain fixed-rate bullet bond and a *parallel* curve shift, z + s enters
the discount factor additively, so CS01 == DV01_curve exactly. The two risks
differ in (a) which *factor* moves (rates vs spread), (b) how much that factor
moves (spread vol >> rate vol for HY / in stress), and (c) non-parallel curve
moves, optionality, and default/recovery. So the useful comparison is
"sensitivity x typical factor move" (risk contribution), not the raw numbers.

Sign convention: positive number = loss for a +1bp move.
"""
from __future__ import annotations

from datetime import date

from scipy.optimize import brentq

from ..pricing.bond import Bond, ZeroCurve, dirty_price_from_curve
from ..pricing.curves import shift_curve

ONE_BP = 1e-4


def cs01(bond: Bond, settlement: date, curve: ZeroCurve, spread: float,
         notional: float = 100.0, bump: float = ONE_BP) -> float:
    """Central-difference loss for +1bp spread move, curve fixed."""
    up = dirty_price_from_curve(bond, settlement, curve, spread + bump)
    dn = dirty_price_from_curve(bond, settlement, curve, spread - bump)
    return (dn - up) / 2.0 * (notional / 100.0) * (ONE_BP / bump)


def dv01_curve(bond: Bond, settlement: date, curve: ZeroCurve, spread: float,
               notional: float = 100.0, bump: float = ONE_BP) -> float:
    """Central-difference loss for +1bp parallel curve move, spread fixed."""
    up = dirty_price_from_curve(bond, settlement, shift_curve(curve, bump), spread)
    dn = dirty_price_from_curve(bond, settlement, shift_curve(curve, -bump), spread)
    return (dn - up) / 2.0 * (notional / 100.0) * (ONE_BP / bump)


def spread_duration(bond: Bond, settlement: date, curve: ZeroCurve, spread: float,
                    bump: float = ONE_BP) -> float:
    """-(1/P) dP/ds, in years."""
    p0 = dirty_price_from_curve(bond, settlement, curve, spread)
    up = dirty_price_from_curve(bond, settlement, curve, spread + bump)
    dn = dirty_price_from_curve(bond, settlement, curve, spread - bump)
    return (dn - up) / (2.0 * p0 * bump)


def z_spread(bond: Bond, settlement: date, dirty_price: float, curve: ZeroCurve) -> float:
    """Constant spread over the zero curve that reprices the bond (Brent)."""
    g = lambda s: dirty_price_from_curve(bond, settlement, curve, s) - dirty_price
    lo, hi = -0.10, 3.0
    if g(lo) * g(hi) > 0:
        raise ValueError("dirty_price is outside the solvable range")
    return float(brentq(g, lo, hi, xtol=1e-14, rtol=1e-14, maxiter=200))


In [ ]:
%%writefile credit_engine_src/credit_engine/data/fred.py
"""FRED download layer.

* Works WITHOUT an API key (public CSV endpoint); an API key is used if given
  (argument or FRED_API_KEY environment variable).
* Every download is cached on disk (default ``data/raw``, git-ignored). Raw
  Moody's / ICE data must not be redistributed, so it is never committed.
* If a refresh fails but an old cache exists, the stale cache is used with a warning.
* The parsing / caching logic is tested offline with mocked HTTP; the real
  network call is verified by running ``notebooks/01_data_check.ipynb``.

Units: FRED spreads and yields are in percent. ``load_series`` converts them to
decimals (4.5 -> 0.045) so they plug directly into the pricing code.
"""
from __future__ import annotations

import io
import os
import time
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
import requests

FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"
FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"
_HEADERS = {"User-Agent": "Mozilla/5.0 (credit-analytics-engine; research use)"}


class FredError(RuntimeError):
    """Raised for download or parsing problems, with a human-readable hint."""


# short name -> (FRED id, unit, description)
CATALOG: dict[str, tuple[str, str, str]] = {
    "baa10y": ("BAA10Y", "percent", "Moody's Baa yield minus 10Y Treasury (daily, from 1986)"),
    "aaa10y": ("AAA10Y", "percent", "Moody's Aaa yield minus 10Y Treasury (daily, from 1986)"),
    "baa_yield": ("DBAA", "percent", "Moody's seasoned Baa corporate bond yield (daily, from 1986)"),
    "aaa_yield": ("DAAA", "percent", "Moody's seasoned Aaa corporate bond yield (daily, from 1983)"),
    "hy_oas": ("BAMLH0A0HYM2", "percent", "ICE BofA US High Yield OAS (FRED keeps ~3y only)"),
    "bbb_oas": ("BAMLC0A4CBBB", "percent", "ICE BofA BBB US Corporate OAS (~3y only)"),
    "aaa_oas": ("BAMLC0A1CAAA", "percent", "ICE BofA AAA US Corporate OAS (~3y only)"),
    "dgs2": ("DGS2", "percent", "2Y Treasury constant maturity yield"),
    "dgs5": ("DGS5", "percent", "5Y Treasury constant maturity yield"),
    "dgs10": ("DGS10", "percent", "10Y Treasury constant maturity yield"),
    "dgs20": ("DGS20", "percent", "20Y Treasury constant maturity yield"),
    "dgs30": ("DGS30", "percent", "30Y Treasury constant maturity yield"),
}


# ------------------------------------------------------------------- parsing
def parse_fred_csv(text: str, series_id: str) -> pd.Series:
    """Parse FRED CSV text (header 'observation_date,ID' or 'DATE,ID').

    Missing observations ('.' or empty) become NaN. Result is sorted, unique index.
    """
    head = text.lstrip()[:200].lower()
    if not head or head.startswith("<"):
        raise FredError(f"{series_id}: empty or non-CSV response (HTML error page?)")
    try:
        df = pd.read_csv(io.StringIO(text))
    except Exception as exc:  # pandas raises several types
        raise FredError(f"{series_id}: could not parse CSV ({exc})") from exc
    if df.shape[1] < 2 or df.empty:
        raise FredError(f"{series_id}: CSV has no data columns/rows")
    dates = pd.to_datetime(df.iloc[:, 0], errors="coerce")
    values = pd.to_numeric(df.iloc[:, 1], errors="coerce")
    s = pd.Series(values.to_numpy(), index=pd.DatetimeIndex(dates), name=series_id)
    s = s[~s.index.isna()]
    s = s[~s.index.duplicated(keep="last")].sort_index()
    s.index.name = "date"
    return s


# ------------------------------------------------------------------ download
_RETRY_STATUS = {429, 500, 502, 503, 504}


def _get_with_retries(url: str, params: dict, timeout: float, retries: int, backoff: float):
    """GET with exponential backoff on timeouts, connection errors and 429/5xx.

    4xx errors such as 400/404 (bad series id / bad key) are NOT retried.
    """
    for attempt in range(retries + 1):
        last = attempt == retries
        try:
            r = requests.get(url, params=params, headers=_HEADERS, timeout=timeout)
            if r.status_code in _RETRY_STATUS and not last:
                time.sleep(backoff * 2**attempt)
                continue
            r.raise_for_status()
            return r
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if last:
                raise
            time.sleep(backoff * 2**attempt)


def _download_csv_text(series_id: str, api_key: str | None, timeout: float,
                       retries: int = 5, backoff: float = 2.0) -> str:
    try:
        if api_key:
            r = _get_with_retries(FRED_API_URL,
                                  {"series_id": series_id, "api_key": api_key, "file_type": "json"},
                                  timeout, retries, backoff)
            obs = r.json().get("observations")
            if obs is None:
                raise FredError(f"{series_id}: API response has no 'observations'")
            lines = [f"observation_date,{series_id}"]
            lines += [f"{o['date']},{o['value']}" for o in obs]
            return "\n".join(lines)
        r = _get_with_retries(FRED_CSV_URL, {"id": series_id}, timeout, retries, backoff)
        return r.text
    except requests.exceptions.RequestException as exc:
        code = getattr(exc.response, "status_code", None) if getattr(exc, "response", None) is not None else None
        hint = ("check the series id" if code in (400, 404)
                else "FRED may be slow: re-run the cell (finished series are cached), or set FRED_API_KEY")
        raise FredError(f"{series_id}: download failed after up to {retries + 1} attempts "
                        f"(HTTP {code}, {type(exc).__name__}); {hint}") from exc


def fetch_fred_series(
    series_id: str,
    *,
    start: str | None = None,
    end: str | None = None,
    cache_dir: str | Path = "data/raw",
    max_age_hours: float = 24.0,
    refresh: bool = False,
    api_key: str | None = None,
    timeout: float = 90.0,
    retries: int = 5,
    backoff: float = 2.0,
) -> pd.Series:
    """Return a FRED series (percent units, NaN for missing days), cached on disk."""
    series_id = series_id.upper()
    api_key = api_key or os.environ.get("FRED_API_KEY")
    cache = Path(cache_dir) / f"{series_id}.csv"
    fresh = cache.exists() and (time.time() - cache.stat().st_mtime) < max_age_hours * 3600

    if fresh and not refresh:
        text = cache.read_text()
    else:
        try:
            text = _download_csv_text(series_id, api_key, timeout, retries, backoff)
            parse_fred_csv(text, series_id)          # validate BEFORE overwriting the cache
            cache.parent.mkdir(parents=True, exist_ok=True)
            cache.write_text(text)
        except FredError as exc:
            if not cache.exists():
                raise
            warnings.warn(f"{exc} -- using stale cache {cache}", stacklevel=2)
            text = cache.read_text()

    s = parse_fred_csv(text, series_id)
    if start or end:
        s = s.loc[start:end]
    return s


def load_series(names: list[str], *, to_decimal: bool = True, on_error: str = "raise", **kwargs) -> pd.DataFrame:
    """Load several catalog series into one DataFrame (outer join on dates).

    on_error="raise": stop at the first failed download (default, strict).
    on_error="skip" : keep going, warn, and record failures in ``df.attrs["failed"]``.
    Finished downloads are cached, so re-running only retries what is missing.
    """
    if on_error not in ("raise", "skip"):
        raise ValueError("on_error must be 'raise' or 'skip'")
    cols: dict[str, pd.Series] = {}
    failed: dict[str, str] = {}
    for name in names:
        if name not in CATALOG:
            raise KeyError(f"Unknown series name {name!r}. Known names: {sorted(CATALOG)}")
        series_id, unit, _ = CATALOG[name]
        try:
            s = fetch_fred_series(series_id, **kwargs)
        except FredError as exc:
            if on_error == "raise":
                raise
            failed[name] = str(exc)
            warnings.warn(f"skipped {name}: {exc}", stacklevel=2)
            continue
        cols[name] = s / 100.0 if (to_decimal and unit == "percent") else s
    if not cols:
        raise FredError("none of the requested series could be downloaded: " + "; ".join(failed.values()))
    df = pd.concat(cols, axis=1, sort=True)
    df.index.name = "date"
    df.attrs["failed"] = failed
    return df


# -------------------------------------------------------------------- checks
def data_report(df: pd.DataFrame, scale: float = 100.0) -> pd.DataFrame:
    """One row per column: first/last date, #obs, min/max/last (in percent if df is decimal)."""
    rows = []
    for c in df.columns:
        s = df[c].dropna()
        if s.empty:
            rows.append({"series": c, "first": None, "last": None, "n_obs": 0,
                         "min_pct": None, "max_pct": None, "last_pct": None})
            continue
        rows.append({"series": c, "first": s.index[0].date(), "last": s.index[-1].date(),
                     "n_obs": len(s), "min_pct": s.min() * scale, "max_pct": s.max() * scale,
                     "last_pct": s.iloc[-1] * scale})
    return pd.DataFrame(rows).set_index("series").round(3)


def check_expectations(df: pd.DataFrame, today=None) -> list[tuple[str, bool, str]]:
    """Sanity checks on what we EXPECT from the data. A FAIL is information, not a crash."""
    today = pd.Timestamp(today if today is not None else date.today())
    rep = data_report(df)
    checks: list[tuple[str, bool, str]] = []
    for name, msg in df.attrs.get("failed", {}).items():
        checks.append((f"{name}: downloaded", False, msg))
    for name in ("baa10y", "aaa10y", "dgs10", "baa_yield", "aaa_yield"):
        if name in rep.index and rep.loc[name, "first"] is not None:
            first = pd.Timestamp(rep.loc[name, "first"])
            checks.append((f"{name}: history starts before 1990", first < pd.Timestamp("1990-01-01"),
                           f"first obs {first.date()}"))
    if "hy_oas" in rep.index and rep.loc["hy_oas", "first"] is not None:
        n = int(rep.loc["hy_oas", "n_obs"])
        checks.append(("hy_oas: about 3 years of data (>= 500 obs)", n >= 500,
                       f"{n} obs, first obs {rep.loc['hy_oas', 'first']} (expected ~2023-09 given FRED's rolling window)"))
    for name in rep.index:
        if rep.loc[name, "last"] is None:
            checks.append((f"{name}: has data", False, "no observations"))
            continue
        lag = (today - pd.Timestamp(rep.loc[name, "last"])).days
        checks.append((f"{name}: last observation within 14 days", lag <= 14,
                       f"last obs {rep.loc[name, 'last']} ({lag} days ago)"))
    return checks


def print_checks(checks: list[tuple[str, bool, str]]) -> None:
    for name, ok, detail in checks:
        print(f"[{'PASS' if ok else 'FAIL'}] {name}  |  {detail}")
    n_fail = sum(1 for _, ok, _ in checks if not ok)
    print(f"\n{len(checks) - n_fail}/{len(checks)} checks passed")


In [ ]:
%%writefile credit_engine_src/credit_engine/spreads/spread_calc.py
"""Spread construction, data-gap detection, and validation of proxy spreads.

Why this module exists: FRED only serves ~3 years of ICE OAS, so long-history
work uses Moody's yield minus a Treasury yield as a PROXY. A proxy is only
useful if we measure how close it is to the real thing on the overlap window.
"""
from __future__ import annotations

import numpy as np
import pandas as pd


def spread(yield_series: pd.Series, benchmark: pd.Series, name: str | None = None) -> pd.Series:
    """yield - benchmark on the dates where both exist (decimals in, decimals out)."""
    df = pd.concat({"y": yield_series, "b": benchmark}, axis=1).dropna()
    out = df["y"] - df["b"]
    out.name = name or "spread"
    return out


def find_gaps(s: pd.Series, min_gap_days: int = 10) -> pd.DataFrame:
    """Calendar gaps between consecutive observations longer than ``min_gap_days``.

    10 days ignores weekends and holidays but catches discontinued series
    (e.g. a Treasury maturity that was not issued for several years).
    """
    idx = pd.DatetimeIndex(s.dropna().index)
    cols = ["last_obs_before", "first_obs_after", "calendar_days"]
    if len(idx) < 2:
        return pd.DataFrame(columns=cols)
    days = np.diff(idx.values).astype("timedelta64[D]").astype(int)
    pos = np.where(days > min_gap_days)[0]
    return pd.DataFrame({cols[0]: idx[pos].date, cols[1]: idx[pos + 1].date, cols[2]: days[pos]})


def gaps_report(df: pd.DataFrame, min_gap_days: int = 10) -> pd.DataFrame:
    """find_gaps for every column; empty frame if no series has a gap."""
    parts = []
    for c in df.columns:
        g = find_gaps(df[c], min_gap_days)
        if not g.empty:
            g.insert(0, "series", c)
            parts.append(g)
    if not parts:
        return pd.DataFrame(columns=["series", "last_obs_before", "first_obs_after", "calendar_days"])
    return pd.concat(parts, ignore_index=True)


def proxy_validation(proxy: pd.Series, reference: pd.Series, freq: str = "W-FRI") -> dict:
    """Compare a long-history proxy spread with a reference OAS on their overlap.

    Reported (spreads in decimals; gaps in bp):
      level_gap_mean_bp / level_gap_std_bp : proxy - reference (a constant gap means a fixed
                                             offset, e.g. maturity/composition difference)
      corr_levels                          : inflated by trends, shown for completeness only
      corr_daily_changes / corr_weekly_changes : do the two move together? (the number that matters)
      beta_weekly                          : reference change per unit of proxy change
                                             (< 1 means the proxy exaggerates moves)
    """
    df = pd.concat({"proxy": proxy, "ref": reference}, axis=1).dropna()
    if len(df) < 60:
        raise ValueError(f"overlap too short ({len(df)} obs); need at least 60")
    gap = df["proxy"] - df["ref"]
    daily = df.diff().dropna()
    weekly = df.resample(freq).last().dropna().diff().dropna()
    if len(weekly) < 20:
        raise ValueError("overlap too short for weekly statistics")
    return {
        "n_days": len(df),
        "overlap_start": df.index[0].date(),
        "overlap_end": df.index[-1].date(),
        "level_gap_mean_bp": gap.mean() * 1e4,
        "level_gap_std_bp": gap.std() * 1e4,
        "corr_levels": df["proxy"].corr(df["ref"]),
        "corr_daily_changes": daily["proxy"].corr(daily["ref"]),
        "corr_weekly_changes": weekly["proxy"].corr(weekly["ref"]),
        "beta_weekly": float(np.polyfit(weekly["proxy"], weekly["ref"], 1)[0]),
    }


In [ ]:
%%writefile credit_engine_src/credit_engine/spreads/events.py
"""Credit-market stress episodes for chart annotation.

Dates are APPROXIMATE and only used to label charts visually. They are not an
input to any model (regime models must not be told the answer).
"""
from __future__ import annotations

EVENTS: list[tuple[str, str, str]] = [
    ("Russia/LTCM 1998", "1998-08-01", "1998-11-30"),
    ("Dot-com credit cycle", "2001-03-01", "2002-10-31"),
    ("Global Financial Crisis", "2007-08-01", "2009-06-30"),
    ("US downgrade / euro crisis", "2011-08-01", "2011-12-31"),
    ("Energy & China 2015-16", "2015-06-01", "2016-02-29"),
    ("COVID-19", "2020-02-20", "2020-04-30"),
    ("Rate shock 2022", "2022-01-01", "2022-10-31"),
    ("SVB 2023", "2023-03-08", "2023-05-15"),
]


def plot_with_events(s_bp, title: str, events=EVENTS, figsize=(12, 5)):
    """Line chart of a spread in bp with shaded stress episodes (skips out-of-range ones)."""
    import matplotlib.pyplot as plt
    import pandas as pd

    s = s_bp.dropna()
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(s.index, s.values, lw=1.0, color="tab:blue")
    top = float(s.max())
    for name, a, b in events:
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        if b < s.index[0] or a > s.index[-1]:
            continue
        ax.axvspan(a, b, color="tab:red", alpha=0.18, lw=0)
        ax.text(a + (b - a) / 2, top, name, rotation=90, va="top", ha="center", fontsize=8, color="darkred")
    ax.set_title(title)
    ax.set_ylabel("bp")
    ax.margins(x=0.01)
    fig.tight_layout()
    return fig


In [ ]:
%%writefile credit_engine_src/credit_engine/spreads/regimes.py
"""Two-regime Markov switching model for credit spreads (calm vs stress).

Model: weekly log-changes of the spread, r_t = mu_s + sigma_s * eps_t, with the hidden state s in {calm, stress}
following a Markov chain. Both the mean AND the variance switch. Spread changes are close to stationary, so this is
statistically far better behaved than fitting the (very persistent) spread level; regimes are identified mainly
through volatility clustering, which is what distinguishes stress episodes. The regime MEANS are statistically
weak (their standard errors are about as large as the means themselves); interpret sigma, durations and time shares.

LOOK-AHEAD (the point of this module): the model returns two probability series.
  * ``stress_prob``           FILTERED  P(stress_t | data up to t)      -> usable live, use this for any signal
  * ``smoothed_stress_prob``  SMOOTHED  P(stress_t | ALL data)          -> uses the future, diagnostics only
Note the parameters are still estimated on the full sample (in-sample). A truly live system must refit on an
expanding window; that is a later step. What is guaranteed here: given the parameters, ``stress_prob`` at time t
never changes when future data are appended (tested).
"""
from __future__ import annotations

import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

CALM, STRESS = "calm", "stress"


def weekly_level_bp(spread: pd.Series, freq: str = "W-FRI") -> pd.Series:
    """Weekly (last observation) spread level in bp from a daily spread in decimals."""
    return (spread.dropna().resample(freq).last().dropna() * 1e4).rename("spread_bp")


def weekly_log_changes(spread: pd.Series, freq: str = "W-FRI") -> pd.Series:
    """Weekly log-changes of a strictly positive spread (decimals in)."""
    s = spread.dropna()
    if (s <= 0).any():
        raise ValueError(f"spread must be strictly positive to take logs (min = {s.min():.6f})")
    w = s.resample(freq).last().dropna()
    return np.log(w).diff().dropna().rename("dlog_spread")


@dataclass(frozen=True)
class RegimeResult:
    stress_prob: pd.Series            # FILTERED: safe to use live
    smoothed_stress_prob: pd.Series   # SMOOTHED: look-ahead, diagnostics only
    summary: pd.DataFrame             # one row per regime
    transition: pd.DataFrame          # P(next | current); rows = from, cols = to
    loglik: float
    aic: float
    bic: float
    converged: bool | None
    params: pd.Series
    param_se: pd.Series               # standard errors from the MLE (statsmodels' res.bse), same index as params
    n_obs: int


def _fit_once(r: pd.Series, search_reps: int, seed: int, k_regimes: int = 2):
    from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

    np.random.seed(seed)                                   # statsmodels draws start values from the global RNG
    mod = MarkovRegression(r, k_regimes=k_regimes, trend="c", switching_variance=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = mod.fit(search_reps=search_reps, disp=False)
    return mod, res


def _fit_with_retries(r: pd.Series, search_reps: int, seed: int, k_regimes: int = 2, max_attempts: int = 8):
    """Fit with a shrinking random search on failure (rare numerical singularities); keeps the best log-likelihood
    seen and prefers a converged result. Shared by ``fit_regime_model`` and ``compare_regime_counts``."""
    best = None
    for attempt in range(max_attempts):
        reps = max(1, search_reps >> attempt)              # shrink the random search on later, more conservative tries
        try:
            _, res = _fit_once(r, reps, seed + 97 * attempt, k_regimes=k_regimes)   # large odd stride avoids seed collisions
        except (np.linalg.LinAlgError, RuntimeError):
            continue                                       # rare numerical singularity; retry with a new seed
        conv = res.mle_retvals.get("converged") if hasattr(res, "mle_retvals") else None
        if best is None or (bool(conv), res.llf) > (bool(best[1]), best[0].llf):
            best = (res, conv)
        if conv:
            break
    if best is None:
        raise RuntimeError(f"regime fit failed on all {max_attempts} attempts (numerical singularity)")
    return best


def fit_regime_model(spread: pd.Series, *, freq: str = "W-FRI", search_reps: int = 20, seed: int = 0,
                     max_attempts: int = 8) -> RegimeResult:
    """Fit the 2-regime model. Retries with new seeds if the optimiser does not converge; keeps the best log-likelihood."""
    r = weekly_log_changes(spread, freq)
    level = weekly_level_bp(spread, freq).reindex(r.index)
    res, conv = _fit_with_retries(r, search_reps, seed, k_regimes=2, max_attempts=max_attempts)

    sig2 = np.array([res.params["sigma2[0]"], res.params["sigma2[1]"]])
    k_stress = int(np.argmax(sig2))                        # label switching: stress = the high-variance state
    k_calm = 1 - k_stress
    filt = res.filtered_marginal_probabilities
    smooth = res.smoothed_marginal_probabilities
    dur = res.expected_durations
    P = np.asarray(res.regime_transition)[:, :, 0]         # P[i, j] = P(S_t = i | S_{t-1} = j)

    rows = {}
    for name, k in ((CALM, k_calm), (STRESS, k_stress)):
        w = filt[k].to_numpy()
        sigma_se = res.bse[f"sigma2[{k}]"] / (2 * np.sqrt(sig2[k])) * 100   # delta method: se(sqrt(x)) ~ se(x)/(2*sqrt(x))
        rows[name] = {
            "weekly_sigma_pct": float(np.sqrt(sig2[k]) * 100),
            "weekly_sigma_se_pct": float(sigma_se),
            "mean_weekly_change_pct": float(res.params[f"const[{k}]"] * 100),
            "mean_se_pct": float(res.bse[f"const[{k}]"] * 100),
            "expected_duration_weeks": float(dur[k]),
            "avg_spread_bp": float(np.average(level.to_numpy(), weights=w)),
            "share_of_time_pct": float(w.mean() * 100),
        }
    order = [k_calm, k_stress]
    trans = pd.DataFrame(P.T[np.ix_(order, order)], index=[f"from {CALM}", f"from {STRESS}"],
                         columns=[f"to {CALM}", f"to {STRESS}"])
    return RegimeResult(
        stress_prob=filt[k_stress].rename("p_stress_filtered"),
        smoothed_stress_prob=smooth[k_stress].rename("p_stress_smoothed"),
        summary=pd.DataFrame(rows).T,
        transition=trans,
        loglik=float(res.llf), aic=float(res.aic), bic=float(res.bic),
        converged=None if conv is None else bool(conv),
        params=res.params, param_se=res.bse, n_obs=len(r),
    )


def regime_periods(prob: pd.Series, threshold: float = 0.5, level_bp: pd.Series | None = None,
                   min_weeks: int = 1) -> pd.DataFrame:
    """Contiguous runs where prob >= threshold: start, end, number of weeks, peak probability (and peak spread)."""
    flag = (prob >= threshold).to_numpy()
    rows, i, n = [], 0, len(prob)
    while i < n:
        if not flag[i]:
            i += 1
            continue
        j = i
        while j + 1 < n and flag[j + 1]:
            j += 1
        seg = prob.iloc[i:j + 1]
        row = {"start": seg.index[0].date(), "end": seg.index[-1].date(), "weeks": j - i + 1,
               "peak_prob": float(seg.max())}
        if level_bp is not None:
            row["peak_spread_bp"] = float(level_bp.reindex(seg.index).max())
        rows.append(row)
        i = j + 1
    cols = ["start", "end", "weeks", "peak_prob"] + (["peak_spread_bp"] if level_bp is not None else [])
    out = pd.DataFrame(rows, columns=cols)
    return out[out["weeks"] >= min_weeks].reset_index(drop=True)


def detection_table(prob: pd.Series, events, threshold: float = 0.5, pad_days: int = 0) -> pd.DataFrame:
    """For each (name, start, end) event: did the FILTERED stress probability cross the threshold, and when?

    Event dates are approximate/subjective, so ``lag_days`` (first alert minus event start) is indicative, not a
    rigorous detection delay. Events outside the sample are skipped.
    """
    rows = []
    for name, a, b in events:
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        if b < prob.index[0] or a > prob.index[-1]:
            continue
        win = prob.loc[a - pd.Timedelta(days=pad_days):b]
        if win.empty:
            continue
        hit = win[win >= threshold]
        rows.append({
            "event": name, "window": f"{a.date()} -> {b.date()}",
            "first_alert": hit.index[0].date() if len(hit) else None,
            "lag_days": (hit.index[0] - a).days if len(hit) else np.nan,
            "weeks_in_stress_pct": float((win >= threshold).mean() * 100),
            "peak_prob": float(win.max()),
        })
    return pd.DataFrame(rows)


def plot_regimes(spread_bp: pd.Series, stress_prob: pd.Series, title: str, events=None, threshold: float = 0.5,
                 figsize=(12, 7)):
    """Top: spread with marked events. Bottom: model's FILTERED stress probability with the same events."""
    import matplotlib.pyplot as plt

    fig, (a1, a2) = plt.subplots(2, 1, figsize=figsize, sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    a1.plot(spread_bp.index, spread_bp.values, lw=1.0, color="tab:blue")
    a1.set_ylabel("bp")
    a1.set_title(title)
    a2.fill_between(stress_prob.index, stress_prob.values, color="tab:orange", alpha=0.6, lw=0)
    a2.axhline(threshold, color="k", lw=0.8, ls="--")
    a2.set_ylim(0, 1)
    a2.set_ylabel("P(stress), filtered")
    if events:
        top = float(spread_bp.max())
        for name, a, b in events:
            a, b = pd.Timestamp(a), pd.Timestamp(b)
            if b < spread_bp.index[0] or a > spread_bp.index[-1]:
                continue
            for ax in (a1, a2):
                ax.axvspan(a, b, color="tab:red", alpha=0.15, lw=0)
            a1.text(a + (b - a) / 2, top, name, rotation=90, va="top", ha="center", fontsize=7, color="darkred")
    fig.tight_layout()
    return fig


def compare_regime_counts(spread: pd.Series, *, freq: str = "W-FRI", k_range: tuple[int, ...] = (2, 3),
                          search_reps: int = 15, seed: int = 0, max_attempts: int = 8) -> pd.DataFrame:
    """Fit the same weekly-log-change series with different numbers of hidden regimes and compare by
    log-likelihood, AIC and BIC (lower AIC/BIC is better; BIC penalises extra parameters more heavily and is
    the more conservative choice for "how many regimes are actually supported by the data").

    A 3-regime model has more parameters (more ways to fit noise) and will always have a higher (or equal)
    log-likelihood than a 2-regime model on the same data; AIC/BIC exist precisely to penalise that extra
    flexibility. If AIC/BIC do NOT improve when moving from 2 to 3 regimes, that is evidence the second regime
    the 3-state model finds is not earning its extra parameters, not evidence that credit spreads only ever
    have two "true" states.
    """
    r = weekly_log_changes(spread, freq)
    rows = []
    for k in k_range:
        res, conv = _fit_with_retries(r, search_reps, seed, k_regimes=k, max_attempts=max_attempts)
        n_params = len(res.params)
        rows.append({"k_regimes": k, "n_params": n_params, "loglik": float(res.llf),
                    "aic": float(res.aic), "bic": float(res.bic), "converged": conv})
    out = pd.DataFrame(rows).set_index("k_regimes")
    out["bic_favors"] = out["bic"].idxmin()
    out["aic_favors"] = out["aic"].idxmin()
    return out


In [ ]:
%%writefile credit_engine_src/credit_engine/spreads/backtest.py
"""Expanding-window refit: simulate what a LIVE system would have seen.

Section E's ``fit_regime_model`` fits parameters on the FULL sample (in-sample). That is fine to learn the
model's structure, but it is optimistic: at any past date t, the in-sample model already "knows" the
parameters that best explain the whole history, including everything after t.

Here the model is refit periodically (e.g. quarterly) using ONLY data available up to each refit date, and the
resulting fixed parameters are used to filter forward until the next refit. Concatenating these pieces gives a
"live" stress-probability series where the value at any date t depends only on parameters estimated from data
strictly before t and on data up to and including t.

NOTE ON REPRODUCIBILITY: the per-refit optimisation (EM + quasi-Newton) is not always bit-identical across
runs even with a fixed seed, because the underlying linear algebra can use multi-threaded BLAS with a
non-deterministic reduction order; this occasionally lands in a different (but statistically equivalent,
label-swapped) local optimum. This does NOT create look-ahead: what is tested and guaranteed is that the
training window for refit i (hence everything the optimiser can possibly see) never includes data at or after
the truncation point of a shorter run of the same series -- see test_backtest.py.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from .regimes import CALM, STRESS, RegimeResult, _fit_once, weekly_log_changes


@dataclass(frozen=True)
class ExpandingResult:
    stress_prob: pd.Series      # "live" filtered stress probability, stitched across refits
    refit_log: pd.DataFrame     # one row per refit: date, n_train_weeks, converged, calm/stress sigma, k_stress
    n_refits: int


def refit_dates(index: pd.DatetimeIndex, min_train_weeks: int, refit_every: int) -> list[pd.Timestamp]:
    """Refit dates: index[min_train_weeks - 1], then every ``refit_every`` weeks after, up to the last date.

    Determined purely by position counted forward from the start, so it does NOT depend on how much data
    exists beyond a given point (needed for the no-lookahead guarantee: see test_backtest.py).
    """
    if len(index) <= min_train_weeks:
        raise ValueError(f"only {len(index)} weeks of data, need > {min_train_weeks} (min_train_weeks)")
    return list(index[min_train_weeks - 1 :: refit_every])


def _fit_once_robust(train: pd.Series, search_reps: int, seed: int, max_attempts: int = 8):
    """_fit_once, retried with new random start values if the optimiser hits a numerical singularity
    (rare with very short/degenerate windows) or fails to converge; keeps the best log-likelihood seen."""
    best = None
    for attempt in range(max_attempts):
        reps = max(1, search_reps >> attempt)
        try:
            _, res = _fit_once(train, reps, seed + 1000 * attempt)
        except (np.linalg.LinAlgError, RuntimeError):
            continue
        conv = res.mle_retvals.get("converged") if hasattr(res, "mle_retvals") else None
        if best is None or (bool(conv), res.llf) > (bool(best.mle_retvals.get("converged")), best.llf):
            best = res
        if conv:
            break
    if best is None:
        raise RuntimeError(f"regime fit failed on all {max_attempts} attempts (numerical singularity)")
    return best


def expanding_refit(spread: pd.Series, *, freq: str = "W-FRI", min_train_weeks: int = 104,
                    refit_every: int = 13, search_reps: int = 10, seed: int = 0) -> ExpandingResult:
    """Refit every ``refit_every`` weeks (13 ~ one quarter) on an expanding window; stitch filtered probabilities.

    ``min_train_weeks`` (default 104 ~ 2 years) is the minimum history before the first fit is attempted.
    """
    from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

    r = weekly_log_changes(spread, freq)
    dates = refit_dates(r.index, min_train_weeks, refit_every)

    pieces: list[pd.Series] = []
    log_rows = []
    for i, d in enumerate(dates):
        train = r.loc[:d]
        res = _fit_once_robust(train, search_reps, seed + i)
        conv = res.mle_retvals.get("converged") if hasattr(res, "mle_retvals") else None
        sig2 = np.array([res.params["sigma2[0]"], res.params["sigma2[1]"]])
        k_stress = int(np.argmax(sig2))

        window_end = dates[i + 1] if i + 1 < len(dates) else r.index[-1]
        window = r.loc[:window_end]
        filt = MarkovRegression(window, k_regimes=2, trend="c", switching_variance=True).filter(res.params)
        piece_start = r.index[0] if i == 0 else d
        piece = filt.filtered_marginal_probabilities[k_stress].loc[piece_start:window_end]
        pieces.append(piece)
        log_rows.append({"refit_date": d.date(), "n_train_weeks": len(train), "converged": conv,
                         "calm_sigma_pct": float(np.sqrt(sig2[1 - k_stress]) * 100),
                         "stress_sigma_pct": float(np.sqrt(sig2[k_stress]) * 100)})

    stitched = pd.concat(pieces)
    stitched = stitched[~stitched.index.duplicated(keep="last")].sort_index().rename("p_stress_live")
    return ExpandingResult(stress_prob=stitched, refit_log=pd.DataFrame(log_rows), n_refits=len(dates))


def compare_live_vs_insample(live: pd.Series, insample: pd.Series) -> dict:
    """How different is the honest 'live' probability from the optimistic in-sample one?

    mean_abs_diff_pp / max_abs_diff_pp: in percentage points of probability.
    agreement_pct: share of weeks where both sides of a 0.5 threshold agree.
    corr: correlation of the two probability series (not just the 0/1 calls).
    """
    df = pd.concat({"live": live, "insample": insample}, axis=1).dropna()
    if df.empty:
        raise ValueError("no overlapping dates between live and in-sample probabilities")
    diff = (df["live"] - df["insample"]).abs()
    agree = ((df["live"] >= 0.5) == (df["insample"] >= 0.5)).mean()
    return {
        "n_weeks": len(df),
        "mean_abs_diff_pp": float(diff.mean() * 100),
        "max_abs_diff_pp": float(diff.max() * 100),
        "agreement_pct": float(agree * 100),
        "corr": float(df["live"].corr(df["insample"])),
    }


def min_train_weeks_sensitivity(spread: pd.Series, insample_stress_prob: pd.Series,
                                candidates: list[int] = (52, 104, 156, 208), *, refit_every: int = 13,
                                search_reps: int = 8, seed: int = 0) -> pd.DataFrame:
    """Re-run the expanding-window simulation for several ``min_train_weeks`` values and compare each to the
    in-sample fit, to check whether the live/in-sample gap (Section 6) shrinks with more starting history."""
    from .backtest import compare_live_vs_insample  # local import: avoids a circular reference at module load

    rows = []
    for w in candidates:
        live = expanding_refit(spread, min_train_weeks=w, refit_every=refit_every, search_reps=search_reps, seed=seed)
        cmp = compare_live_vs_insample(live.stress_prob, insample_stress_prob)
        rows.append({"min_train_weeks": w, "n_refits": live.n_refits, **cmp})
    return pd.DataFrame(rows).set_index("min_train_weeks")


In [ ]:
%%writefile credit_engine_src/credit_engine/portfolio/risk_contribution.py
"""Risk contribution: sensitivity x typical factor move, and total risk WITH correlation.

Section 2.3's finding was that CS01 equals curve DV01 exactly for a parallel shift on a plain bullet bond
(tested in test_cs01.py). That means the two risk MEASURES cannot be compared directly -- what differs between
rate risk and credit risk is how much each FACTOR typically moves, not the bond's sensitivity to it.

This module makes that comparison explicit and quantitative:

    weekly_risk_contribution = sensitivity_per_bp * typical_weekly_move_in_bp

where "typical move" is the realized weekly standard deviation of the factor (Treasury yield for rate risk,
credit spread for spread risk), estimated from real FRED data -- not assumed.

IMPORTANT -- correlation: ``rate_risk_contribution`` and ``spread_risk_contribution`` are MARGINAL pieces; they
do not sum to the portfolio's true P&L variance unless rate and spread moves are uncorrelated. In reality they
usually are not (e.g. flight-to-quality: rates often fall while spreads widen in a crisis, a NEGATIVE
correlation). The correct total is

    Var(P&L) = DV01^2 * sigma_r^2 + CS01^2 * sigma_s^2 + 2 * DV01 * CS01 * rho * sigma_r * sigma_s

which is what ``total_risk_1sd`` computes (a one-standard-deviation $ P&L swing in a typical week). Ignoring
rho and just adding the two marginal contributions overstates risk when rho < 0 and understates it when rho > 0.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import numpy as np
import pandas as pd

from ..pricing.bond import Bond
from ..pricing.curves import ZeroCurve, flat_curve
from ..risk.cs01 import cs01 as _cs01
from ..risk.cs01 import dv01_curve as _dv01_curve


def realized_weekly_vol_bp(series: pd.Series, freq: str = "W-FRI", window_years: float | None = None) -> float:
    """Realized weekly volatility of a yield/spread series, in bp. Input in decimals (0.045 = 4.5%).

    ``window_years``, when given, restricts the calculation to the most recent N years before resampling.
    This matters a lot for a "current risk" report: e.g. the full history of DGS10 (since 1962) includes the
    Volcker-era early-1980s spike, which would badly overstate today's rate volatility if left in. Without a
    window the ENTIRE available history is used, which is appropriate for a long series only if that is what
    you actually want (e.g. for methodological comparisons); for a live risk view, pass a window (e.g. 2-3
    years) so both the rate and the spread volatility describe the SAME, recent regime.
    """
    s = series.dropna()
    if window_years is not None:
        cutoff = s.index.max() - pd.Timedelta(days=window_years * 365.25)
        s = s[s.index >= cutoff]
    if len(s) < 10:
        raise ValueError(f"need at least 10 observations in the window, got {len(s)}")
    weekly = s.resample(freq).last().dropna()
    changes_bp = weekly.diff().dropna() * 1e4
    if len(changes_bp) < 5:
        raise ValueError(f"need at least 5 weekly changes in the window, got {len(changes_bp)}")
    return float(changes_bp.std())


def factor_correlation(rate_series: pd.Series, spread_series: pd.Series, freq: str = "W-FRI",
                       window_years: float | None = None) -> float:
    """Correlation of WEEKLY CHANGES between a rate series and a spread series (both decimals).

    Computed on changes, not levels: level correlation is inflated by shared trends and says little about how
    the two factors co-move week to week, which is what the portfolio-variance formula needs.
    """
    r, s = rate_series.dropna(), spread_series.dropna()
    if window_years is not None:
        cutoff = min(r.index.max(), s.index.max()) - pd.Timedelta(days=window_years * 365.25)
        r, s = r[r.index >= cutoff], s[s.index >= cutoff]
    rw = r.resample(freq).last().diff().dropna()
    sw = s.resample(freq).last().diff().dropna()
    common = rw.index.intersection(sw.index)
    if len(common) < 10:
        raise ValueError(f"need at least 10 overlapping weekly changes, got {len(common)}")
    return float(rw[common].corr(sw[common]))


@dataclass(frozen=True)
class RiskContribution:
    label: str
    notional: float
    dv01: float                    # $ loss per 1bp parallel rate move (curve-consistent: risk.cs01.dv01_curve)
    cs01: float                    # $ loss per 1bp spread move (== dv01 for a parallel shift, by construction)
    rate_vol_bp: float             # realized weekly Treasury yield vol, bp
    spread_vol_bp: float           # realized weekly credit spread vol, bp
    rho: float                     # correlation of weekly rate and spread changes
    rate_risk_contribution: float  # dv01 * rate_vol_bp: MARGINAL piece, ignores correlation
    spread_risk_contribution: float  # cs01 * spread_vol_bp: MARGINAL piece, ignores correlation
    total_risk_1sd: float          # correlation-adjusted total: sqrt(Var(P&L)), see module docstring
    naive_sum_1sd: float           # rate_risk_contribution + spread_risk_contribution, for comparison only
    dominant_factor: str           # "rate" or "spread": whichever MARGINAL contribution is larger


def risk_contribution_report(
    label: str, bond: Bond, settlement: date, curve: ZeroCurve, spread: float, notional: float,
    rate_series: pd.Series, spread_series: pd.Series, window_years: float | None = 3.0,
) -> RiskContribution:
    """Build the DV01/CS01-vs-realized-vol comparison for one bond, including the correlation-adjusted total.

    ``rate_series`` and ``spread_series`` are decimal-yield time series and should be RATING-APPROPRIATE:
    using the same broad-market spread series for every bond regardless of its credit quality makes
    ``dominant_factor`` identical across all bonds by construction (their DV01==CS01, so the split percentage
    reduces to spread_vol / (rate_vol + spread_vol), independent of the bond, if the same two series are
    reused). Pass e.g. ICE AAA/BBB/HY OAS for the matching rating bucket, not one shared proxy.

    ``curve`` should be the REAL (e.g. Treasury) curve, not necessarily flat: DV01 is computed with
    ``risk.cs01.dv01_curve`` (a curve bump), consistent with how CS01 is computed, so both sensitivities are
    correct even when the curve has a realistic slope.

    ``window_years`` (default 3, matching what FRED's ICE OAS series actually cover) restricts both
    volatilities AND the correlation to the same recent window, so the comparison describes TODAY's regime
    rather than a multi-decade average that can include very different periods (e.g. the early-1980s rate
    spike). Pass None to use the full history of each series instead.
    """
    d = _dv01_curve(bond, settlement, curve, spread, notional=notional)
    c = _cs01(bond, settlement, curve, spread, notional=notional)
    rv = realized_weekly_vol_bp(rate_series, window_years=window_years)
    sv = realized_weekly_vol_bp(spread_series, window_years=window_years)
    rho = factor_correlation(rate_series, spread_series, window_years=window_years)
    rate_contrib = d * rv
    spread_contrib = c * sv
    total = float(np.sqrt(max(0.0, d**2 * rv**2 + c**2 * sv**2 + 2 * d * c * rho * rv * sv)))
    return RiskContribution(
        label=label, notional=notional, dv01=d, cs01=c, rate_vol_bp=rv, spread_vol_bp=sv, rho=rho,
        rate_risk_contribution=rate_contrib, spread_risk_contribution=spread_contrib,
        total_risk_1sd=total, naive_sum_1sd=rate_contrib + spread_contrib,
        dominant_factor="spread" if spread_contrib > rate_contrib else "rate",
    )


# Only 3 ICE OAS buckets are available from FRED (AAA, BBB, HY). For ratings outside these three, use the
# nearest bucket's realized volatility as an approximation -- documented here, not hidden in notebook code.
RATING_TO_ICE_BUCKET: dict[str, str] = {
    "AAA": "aaa_oas", "AA": "aaa_oas", "A": "aaa_oas",     # investment-grade, close to AAA behaviour
    "BBB": "bbb_oas",                                       # lowest investment grade, its own bucket
    "BB (HY)": "hy_oas", "B (HY)": "hy_oas", "CCC (HY)": "hy_oas",  # all high-yield -> the HY bucket
}


def spread_series_for_rating(rating: str, series_by_bucket: dict[str, pd.Series]) -> pd.Series:
    """Look up the ICE OAS series (already fetched into ``series_by_bucket``, e.g. {"aaa_oas": df["aaa_oas"],
    "bbb_oas": df["bbb_oas"], "hy_oas": df["hy_oas"]}) for the given rating, via RATING_TO_ICE_BUCKET."""
    if rating not in RATING_TO_ICE_BUCKET:
        raise KeyError(f"unknown rating {rating!r}; expected one of {sorted(RATING_TO_ICE_BUCKET)}")
    bucket = RATING_TO_ICE_BUCKET[rating]
    if bucket not in series_by_bucket:
        raise KeyError(f"rating {rating!r} maps to {bucket!r}, which is missing from series_by_bucket")
    return series_by_bucket[bucket]


def rating_vol_ratio(rating: str, series_by_bucket: dict[str, pd.Series], reference_series: pd.Series,
                     window_years: float | None = 3.0) -> float:
    """How much more (or less) volatile is this rating's spread than the reference (e.g. Baa-30Y proxy)?

    Used to scale a historical widening scenario sized on the long-history proxy down/up to a specific rating
    bucket (see portfolio.scenarios.scaled_historical_scenarios): HY spreads widen far more than AAA spreads in
    a crisis, so applying the SAME bp widening to every bond is unrealistic (see Section H's original bug).
    """
    bucket_series = spread_series_for_rating(rating, series_by_bucket)
    bucket_vol = realized_weekly_vol_bp(bucket_series, window_years=window_years)
    ref_vol = realized_weekly_vol_bp(reference_series, window_years=window_years)
    return bucket_vol / ref_vol


def report_table(reports: list[RiskContribution]) -> pd.DataFrame:
    rows = []
    for r in reports:
        rows.append({
            "bond": r.label, "notional": r.notional, "dv01_$": r.dv01, "cs01_$": r.cs01,
            "rate_vol_bp": r.rate_vol_bp, "spread_vol_bp": r.spread_vol_bp, "rho": r.rho,
            "rate_contribution_$": r.rate_risk_contribution, "spread_contribution_$": r.spread_risk_contribution,
            "naive_sum_1sd_$": r.naive_sum_1sd, "total_risk_1sd_$": r.total_risk_1sd,
            "dominant": r.dominant_factor,
            "spread_share_pct": 100 * r.spread_risk_contribution / (r.rate_risk_contribution + r.spread_risk_contribution),
        })
    return pd.DataFrame(rows).set_index("bond")


In [ ]:
%%writefile credit_engine_src/credit_engine/portfolio/scenarios.py
"""Stress scenarios: reprice a portfolio of bonds under a curve/spread shock and report P&L.

Three built-in scenario families:
  * parallel(bp)                  -- flat shift of the whole curve
  * steepen(short_bp, long_bp)    -- short end and long end move by different amounts (linear in between)
  * spread_widening(bp)           -- credit spread widens (or tightens, negative bp), curve unchanged (uniform
                                     across every bond -- see the WARNING below)
  * scaled_historical_scenarios   -- spread widening sized PER RATING BUCKET, not uniformly (see below)

Each scenario reprices every bond exactly (via dirty_price_from_curve), not via a duration approximation,
so results are exact even for large shocks where convexity would make a linear DV01 estimate misleading.

WARNING about uniform bp widening: applying the SAME bp spread widening to every bond regardless of its rating
is unrealistic. In a real crisis, high-yield spreads widen far more than AAA spreads (e.g. in 2008, HY OAS
widened by several thousand bp while AAA widened by a few hundred). ``spread_widening`` and
``historical_style_scenarios`` do this uniformly and are kept for simple, single-bucket analyses; for a
multi-rating portfolio, use ``scaled_historical_scenarios`` instead, which scales the widening per bond using
``portfolio.risk_contribution.rating_vol_ratio`` (each bond's own rating volatility relative to the reference
series the widening was measured on).
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date

import numpy as np
import pandas as pd

from ..pricing.bond import Bond, ZeroCurve, dirty_price_from_curve
from ..pricing.curves import linear_curve

SpreadShift = float | dict[str, float]  # uniform (bp, applied to every bond) or per-bond ({label: bp})


@dataclass(frozen=True)
class PortfolioBond:
    label: str
    bond: Bond
    spread: float          # credit spread over the base curve, decimal
    notional: float        # par notional


@dataclass(frozen=True)
class Scenario:
    name: str
    curve_shift: "callable"      # years -> additional decimal shift to the base curve
    spread_shift_bp: SpreadShift  # additional spread shift, in bp: a single float (uniform) or {label: bp}


def _shift_for(spread_shift_bp: SpreadShift, label: str) -> float:
    """Resolve the bp shift for one bond, whether the scenario carries a uniform value or a per-bond dict."""
    if isinstance(spread_shift_bp, dict):
        if label not in spread_shift_bp:
            raise KeyError(f"scenario has no spread shift for bond {label!r}; has: {sorted(spread_shift_bp)}")
        return spread_shift_bp[label]
    return spread_shift_bp


def parallel(bp: float) -> Scenario:
    return Scenario(f"Parallel {bp:+.0f}bp", lambda years: np.full_like(np.asarray(years, dtype=float), bp / 1e4), 0.0)


def steepen(short_bp: float, long_bp: float, pivot_years: tuple[float, float] = (2.0, 30.0)) -> Scenario:
    """Linear interpolation of the shift between ``pivot_years``; flat beyond the pivots."""
    shift_curve = linear_curve(list(pivot_years), [short_bp / 1e4, long_bp / 1e4])
    name = f"Steepener (2Y {short_bp:+.0f}bp / 30Y {long_bp:+.0f}bp)"
    return Scenario(name, shift_curve, 0.0)


def spread_widening(bp: float, label: str | None = None) -> Scenario:
    """Uniform widening: every bond's spread moves by the SAME bp. See the module WARNING above."""
    name = label or f"Spread {bp:+.0f}bp"
    return Scenario(name, lambda years: np.zeros_like(np.asarray(years, dtype=float)), bp)


def apply_scenario(base_curve: ZeroCurve, scenario: Scenario) -> ZeroCurve:
    return lambda years: np.asarray(base_curve(years), dtype=float) + np.asarray(scenario.curve_shift(years), dtype=float)


def price_portfolio(portfolio: list[PortfolioBond], settlement: date, curve: ZeroCurve,
                    spread_bump: SpreadShift = 0.0) -> float:
    """Total dirty market value of the portfolio (per-bond price/100 * notional), summed.

    ``spread_bump`` (bp) is either one number applied to every bond, or a {label: bp} dict for per-bond bumps.
    """
    total = 0.0
    for pb in portfolio:
        bump = _shift_for(spread_bump, pb.label)
        price = dirty_price_from_curve(pb.bond, settlement, curve, pb.spread + bump / 1e4)
        total += price / 100.0 * pb.notional
    return total


def run_scenarios(portfolio: list[PortfolioBond], settlement: date, base_curve: ZeroCurve,
                  scenarios: list[Scenario]) -> pd.DataFrame:
    """P&L (exact reprice, not duration-approximated) for the whole portfolio and per-bond, for each scenario."""
    base_value = price_portfolio(portfolio, settlement, base_curve)
    per_bond_base = {pb.label: dirty_price_from_curve(pb.bond, settlement, base_curve, pb.spread) / 100.0 * pb.notional
                     for pb in portfolio}

    rows = []
    for sc in scenarios:
        shocked_curve = apply_scenario(base_curve, sc)
        shocked_value = price_portfolio(portfolio, settlement, shocked_curve, sc.spread_shift_bp)
        row = {"scenario": sc.name, "portfolio_pnl_$": shocked_value - base_value,
              "portfolio_pnl_pct": (shocked_value - base_value) / base_value * 100}
        for pb in portfolio:
            bump = _shift_for(sc.spread_shift_bp, pb.label)
            shocked_bond_value = dirty_price_from_curve(pb.bond, settlement, shocked_curve,
                                                        pb.spread + bump / 1e4) / 100.0 * pb.notional
            row[f"{pb.label}_pnl_$"] = shocked_bond_value - per_bond_base[pb.label]
        rows.append(row)
    return pd.DataFrame(rows).set_index("scenario")


def historical_style_scenarios(peak_spread_bp: dict[str, float], current_spread_bp: float) -> list[Scenario]:
    """Build UNIFORM spread-widening scenarios sized to historical episodes, e.g. {"2008 GFC": 620,
    "COVID-19": 430} (peak Baa-30Y spread in bp, from regimes.regime_periods) vs. today's level.

    See the module WARNING: this applies the SAME bp widening to every bond regardless of rating. Prefer
    ``scaled_historical_scenarios`` for a multi-rating portfolio.
    """
    return [spread_widening(peak - current_spread_bp, label=f"{name}-style widening (to {peak:.0f}bp)")
           for name, peak in peak_spread_bp.items()]


def scaled_historical_scenarios(peak_spread_bp: dict[str, float], current_spread_bp: float,
                                rating_ratios: dict[str, float]) -> list[Scenario]:
    """Historical-style widening, scaled PER BOND by ``rating_ratios`` (label -> volatility ratio vs. the
    reference series the peaks were measured on, e.g. from ``risk_contribution.rating_vol_ratio``).

    The base widening (peak - current, on the long-history reference proxy) is multiplied by each bond's own
    ratio, so a high-yield bond (ratio > 1, more volatile than the broad reference) widens by MORE than the
    reference figure and an AAA bond (ratio < 1) widens by LESS -- unlike ``historical_style_scenarios``, which
    would apply the same raw widening to both.
    """
    base_widenings = {name: peak - current_spread_bp for name, peak in peak_spread_bp.items()}
    scenarios = []
    for name, base_bp in base_widenings.items():
        per_bond_bp = {label: base_bp * ratio for label, ratio in rating_ratios.items()}
        scenarios.append(Scenario(f"{name}-style widening (scaled by rating)", 
                                  lambda years: np.zeros_like(np.asarray(years, dtype=float)), per_bond_bp))
    return scenarios


## A3. Load and verify
Version should read `0.8.0`. If a submodule is missing (e.g. `ModuleNotFoundError: credit_engine.data.fred`), this cell says so **explicitly** and tells you what to do, instead of a cryptic error.

In [ ]:
import sys, os, importlib

FIX = "\n\n>>> FIX: Runtime > Restart session, then Runtime > Run all to rerun EVERYTHING from the top (not cell by cell)."

sys.path.insert(0, os.path.abspath("credit_engine_src"))
importlib.invalidate_caches()
try:
    import credit_engine
except ModuleNotFoundError as e:
    raise RuntimeError(f"credit_engine could not be imported at all ({e})." + FIX) from e

print("Version:", credit_engine.__version__)
print("Loaded from:", credit_engine.__file__)
if credit_engine.__version__ != "0.8.0" or "credit_engine_src" not in credit_engine.__file__:
    raise RuntimeError(
        f"Unexpected version/location: {credit_engine.__version__} @ {credit_engine.__file__} "
        f"(expected: 0.8.0, inside credit_engine_src)." + FIX)

# Try each submodule individually so we can name exactly which one is missing (the top-level import above can
# succeed even if a submodule failed to get written, since "import credit_engine" only needs the package itself).
_submodules = ["pricing.bond", "pricing.daycount", "pricing.curves", "pricing.yield_solver",
              "risk.duration", "risk.cs01", "data.fred", "spreads.spread_calc", "spreads.events",
              "spreads.regimes", "spreads.backtest", "portfolio.risk_contribution", "portfolio.scenarios"]
_missing = []
for _m in _submodules:
    try:
        importlib.import_module(f"credit_engine.{_m}")
    except ModuleNotFoundError:
        _missing.append(_m)
if _missing:
    raise RuntimeError(f"These submodules are missing: {_missing}. Likely a write cell in A2 was skipped, "
                       f"or an old set of files is still on disk." + FIX)

print(f"All submodules ({len(_submodules)}) loaded. Everything is fine.")

# SECTION B: Day 1, pricing and risk
## B1. Demo: three example bonds
Expected: DV01 = CS01 (you'll see why in later sections). Values: AAA-like 4,352 / BBB-like 7,569 / HY-like 5,119.
Note: the Treasury yield (4.20%) here is just an illustrative made-up number.

In [ ]:
from datetime import date

from credit_engine.pricing.bond import Bond
from credit_engine.pricing.curves import flat_curve
from credit_engine.pricing.bond import dirty_price_from_curve
from credit_engine.risk.cs01 import cs01, dv01_curve, z_spread
from credit_engine.risk.duration import rate_risk_report

SETTLE = date(2026, 9, 21)
TSY = 0.042  # flat illustrative Treasury zero rate

# (label, coupon, maturity, spread over Treasuries) - illustrative numbers only
BONDS = [
    ("AAA-like  5y", Bond(0.045, date(2031, 9, 15)), 0.0060),
    ("BBB-like 10y", Bond(0.058, date(2036, 9, 15)), 0.0150),
    ("HY-like   7y", Bond(0.085, date(2033, 9, 15)), 0.0450),
]

print(f"Settlement {SETTLE}, flat Treasury zero {TSY:.2%}, notional 10mm\n")
print(f"{'bond':<14}{'YTM':>7}{'clean':>9}{'MacD':>7}{'ModD':>7}{'Conv':>8}{'DV01($)':>10}{'CS01($)':>10}")
for label, bond, spread in BONDS:
    curve = flat_curve(TSY)
    dirty = dirty_price_from_curve(bond, SETTLE, curve, spread)
    ytm = TSY + spread
    r = rate_risk_report(bond, SETTLE, ytm, notional=10_000_000)
    assert abs(r.dirty_price - dirty) < 1e-9
    assert abs(z_spread(bond, SETTLE, dirty, curve) - spread) < 1e-10
    c = cs01(bond, SETTLE, curve, spread, notional=10_000_000)
    print(f"{label:<14}{ytm:>7.2%}{r.clean_price:>9.3f}{r.macaulay:>7.3f}{r.modified:>7.3f}"
          f"{r.convexity:>8.2f}{r.dv01:>10,.0f}{c:>10,.0f}")


## B2. Hand-calculation check
A 2-year, 5% annual coupon, par bond. The results can be worked out by hand on paper; the code should match exactly.

In [ ]:
from datetime import date
from credit_engine.pricing.bond import Bond, dirty_price
from credit_engine.risk.duration import macaulay_duration, modified_duration, convexity, dv01

b, s = Bond(0.05, date(2027, 1, 15), frequency=1), date(2025, 1, 15)
checks = [("Price (par)",          dirty_price(b, s, 0.05),        100.0),
          ("Macaulay duration",   macaulay_duration(b, s, 0.05),  1.952381),
          ("Modified duration",   modified_duration(b, s, 0.05),  1.859410),
          ("Convexity",           convexity(b, s, 0.05),          5.269409),
          ("DV01 (100 notional)", dv01(b, s, 0.05),               0.018594)]
for name, got, want in checks:
    print(f"[{'PASS' if abs(got - want) < 1e-5 else 'FAIL'}] {name}: code = {got:.6f} | by hand = {want:.6f}")

## B3. Cross-check against QuantLib (optional but important)
Prices the same bonds with an independent library and compares. Installing it can take ~1 minute. If it can't be installed, the cell says so and you can skip it.

In [ ]:
import subprocess, sys
from datetime import date
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "QuantLib"], capture_output=True)
try:
    import QuantLib as ql
except ImportError:
    ql = None
    print("QuantLib could not be installed; you can skip this cell (it's not required).")

if ql is not None:
    from credit_engine.pricing.bond import Bond, clean_price
    from credit_engine.pricing.daycount import DayCount
    from credit_engine.risk.duration import macaulay_duration, modified_duration, convexity

    def ql_metrics(bond, settle, y):
        ql.Settings.instance().evaluationDate = ql.Date(settle.day, settle.month, settle.year)
        m = bond.maturity
        freq = {1: ql.Annual, 2: ql.Semiannual, 4: ql.Quarterly}[bond.frequency]
        sched = ql.Schedule(ql.Date(m.day, m.month, m.year - 40), ql.Date(m.day, m.month, m.year), ql.Period(freq),
                            ql.NullCalendar(), ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Backward, False)
        dc = (ql.Thirty360(ql.Thirty360.USA) if bond.day_count is DayCount.THIRTY_360
              else ql.ActualActual(ql.ActualActual.ISMA, sched))
        qb = ql.FixedRateBond(0, 100.0, sched, [bond.coupon_rate], dc)
        r = ql.InterestRate(y, dc, ql.Compounded, freq)
        return {"clean": qb.cleanPrice(y, dc, ql.Compounded, freq), "accrued": qb.accruedAmount(),
                "macaulay": ql.BondFunctions.duration(qb, r, ql.Duration.Macaulay),
                "modified": ql.BondFunctions.duration(qb, r, ql.Duration.Modified),
                "convexity": ql.BondFunctions.convexity(qb, r)}

    samples = [("BBB-like 10y 30/360", Bond(0.058, date(2036, 9, 15)), date(2026, 9, 21), 0.057),
               ("HY-like 7y 30/360",  Bond(0.085, date(2033, 9, 15)), date(2026, 9, 21), 0.087),
               ("AAA-like 5y ACT/ACT", Bond(0.045, date(2031, 9, 15), 2, DayCount.ACT_ACT_ICMA), date(2026, 9, 21), 0.048)]
    worst = 0.0
    print(f"{'bond':<22}{'metric':<11}{'ours':>14}{'QuantLib':>14}{'diff':>10}")
    for label, bond, st, y in samples:
        q = ql_metrics(bond, st, y)
        ours = {"clean": clean_price(bond, st, y), "accrued": bond.accrued_interest(st),
                "macaulay": macaulay_duration(bond, st, y), "modified": modified_duration(bond, st, y),
                "convexity": convexity(bond, st, y)}
        for k in ours:
            diff = abs(ours[k] - q[k]); worst = max(worst, diff)
            print(f"{label:<22}{k:<11}{ours[k]:>14.8f}{q[k]:>14.8f}{diff:>10.1e}")
    print(f"\n[{'PASS' if worst < 1e-6 else 'FAIL'}] largest difference: {worst:.1e}")

# SECTION C: Real data (FRED)
## C1. API key (from Colab Secrets)
If there's no key, this warns and continues without one (can be slower, you may see timeouts).

In [ ]:
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None
    print("Looks like you're outside Colab; Secrets aren't available. Continuing without a key.")

if userdata is not None:
    try:
        os.environ["FRED_API_KEY"] = userdata.get("FRED_API_KEY")
        print("FRED API key loaded from Colab Secrets.")
    except userdata.SecretNotFoundError:
        msg = ("PROBLEM: no secret named 'FRED_API_KEY' was found. "
               "Click the key icon in the left panel, choose 'Add new secret'. "
               "Type the Name EXACTLY as 'FRED_API_KEY' (watch case and spaces). "
               "Continuing without a key; this can be slow.")
        print(msg)
    except userdata.NotebookAccessError:
        msg = ("PROBLEM: the 'FRED_API_KEY' secret exists but access for THIS NOTEBOOK is off. "
               "Click the key icon in the left panel, find FRED_API_KEY. "
               "Turn ON 'Notebook access' (a per-notebook permission, off by default). "
               "Continuing without a key; this can be slow.")
        print(msg)
    except Exception as exc:
        print(f"Could not read the API key (unexpected error: {type(exc).__name__}: {exc}). Continuing without a key.")

## C2. Download the data
If a series fails to download, rerun this cell (whatever already succeeded is cached).

In [ ]:
from credit_engine.data.fred import load_series, data_report, check_expectations, print_checks

names = ["baa10y", "aaa10y", "baa_yield", "aaa_yield", "hy_oas", "bbb_oas", "aaa_oas",
         "dgs2", "dgs5", "dgs10", "dgs20", "dgs30"]
df = load_series(names, on_error="skip")
if df.attrs["failed"]:
    print("SERIES THAT DID NOT DOWNLOAD (rerun this cell):", list(df.attrs["failed"]))
data_report(df)                                     # values are in percentage points

## C3. Automated checks
FAIL = information, not an error. **Send me this.**

In [ ]:
print_checks(check_expectations(df))

# SECTION D: Spread analysis
## D1. Data gaps
Is there a long break in the middle of any series? An empty table means no gaps. **Send me this.**

In [ ]:
from credit_engine.spreads.spread_calc import gaps_report

gaps = gaps_report(df)
print("No long gaps." if gaps.empty else f"{len(gaps)} long gap(s) found:")
gaps

## D2. Compute the spreads ourselves
Moody's yield minus Treasury yield (10, 20, 30 year). Our own Baa-10Y calculation should match what FRED publishes directly (difference under ~1bp). **Send me this.**

In [ ]:
import pandas as pd
from credit_engine.spreads.spread_calc import spread

sp = {}
for grade in ("baa", "aaa"):
    for tenor in ("10", "20", "30"):
        y, b = f"{grade}_yield", f"dgs{tenor}"
        if y in df.columns and b in df.columns:
            sp[f"{grade}_{tenor}y"] = spread(df[y], df[b], f"{grade}_{tenor}y")

for k, v in sp.items():
    print(f"{k}: {v.index[0].date()} -> {v.index[-1].date()}, {len(v)} days")

print()
for ours, fred_col in (("baa_10y", "baa10y"), ("aaa_10y", "aaa10y")):
    if ours in sp and fred_col in df.columns:
        d = (sp[ours] - df[fred_col]).dropna().abs() * 1e4
        print(f"{ours} vs FRED {fred_col}: {len(d)} days, mean diff {d.mean():.2f} bp, max diff {d.max():.2f} bp")

## D3. Proxy validation (the most important step)
How close is the long-history Moody's proxy to the real ICE OAS? Measured on the shared ~3-year overlap. **Send me this.**
- `level_gap_mean_bp`: level difference (a constant gap means a maturity/composition difference)
- `corr_weekly_changes`: do weekly changes move together? (the number that actually matters)
- `beta_weekly`: below 1 means the proxy exaggerates moves

In [ ]:
from credit_engine.spreads.spread_calc import proxy_validation

pairs = [("baa_10y", "bbb_oas"), ("baa_20y", "bbb_oas"), ("baa_30y", "bbb_oas"),
         ("aaa_10y", "aaa_oas"), ("aaa_20y", "aaa_oas"), ("aaa_30y", "aaa_oas")]
rows = {}
for k, ref in pairs:
    if k in sp and ref in df.columns:
        rows[f"{k} vs ICE {ref}"] = proxy_validation(sp[k], df[ref])
val = pd.DataFrame(rows).T
print("Overlap window:", val["overlap_start"].iloc[0], "->", val["overlap_end"].iloc[0], f"({int(val['n_days'].iloc[0])} days)")
val.drop(columns=["overlap_start", "overlap_end", "n_days"]).astype(float).round(3)

## D4. Charts
1) The Baa spread, with stress episodes marked (2008 and 2020 should stand out). 2) How close are the proxies to ICE BBB OAS? **Send me this.**

In [ ]:
import matplotlib.pyplot as plt
from credit_engine.spreads.events import plot_with_events

if "baa10y" in df.columns:
    plot_with_events(df["baa10y"].dropna() * 1e4, "Moody's Baa - 10Y Treasury (bp), stress episodes marked")
    plt.show()

cand = {"ICE BBB OAS (reference)": df.get("bbb_oas"), "Baa - 10Y": sp.get("baa_10y"),
        "Baa - 20Y": sp.get("baa_20y"), "Baa - 30Y": sp.get("baa_30y")}
cand = {k: v for k, v in cand.items() if v is not None}
ov = pd.concat(cand, axis=1).dropna() * 1e4
ov.plot(figsize=(12, 5), title="How close are the proxies to ICE BBB OAS? (bp, shared ~3y)")
plt.show()

# SECTION E: Regime model (calm / stress)
**Decision (based on D3):** the primary proxy is **Baa - 30Y**. Almost as good as Baa-20Y, but 20Y has a gap between 1987-1993. Baa-10Y came out weak (52bp level gap, beta 0.68) and is only used as a robustness check. The AAA proxies are weak too (beta 0.4-0.6), so they aren't used.

**Model:** *weekly log-changes* of the spread, 2 hidden regimes (calm / stress), each with its own mean and variance. Stress regime = the high-variance regime.

**Critical rule:** only the **filtered** probability is used (at time t, using only data up to that day). The *smoothed* probability looks into the future and is banned as a signal (tested: the difference can be as large as 0.8).
Note: parameters are estimated on the full sample (in-sample). A live system needs an expanding-window refit -- that's the next section.

## E1. Fit the model and summarize
**Send me this.** The `stress` row is the high-variance regime. Converged should read True. **Careful:** `mean_weekly_change_pct` (the regime means) is statistically unreliable -- the table now also shows `mean_se_pct` and `weekly_sigma_se_pct` (standard errors from the fit): compare each mean to its own SE and you'll usually see they're close in size (not statistically significant), unlike the volatilities, which are precisely estimated. Base your interpretation on `weekly_sigma_pct`, duration and time share instead.

In [ ]:
from credit_engine.spreads.regimes import (fit_regime_model, regime_periods, detection_table,
                                            plot_regimes, weekly_level_bp)
from credit_engine.spreads.events import EVENTS

assert sp["baa_30y"].min() > 0, f"Baa-30Y must be positive (we take logs). Min value: {sp['baa_30y'].min()} -> tell me"
prim = fit_regime_model(sp["baa_30y"])
print("Converged:", prim.converged, "| observations (weeks):", prim.n_obs, "| log-likelihood:", round(prim.loglik, 1))
prim.summary.round(2)

## E2. Transition probabilities
**Send me this.** Row = current regime, column = next week. A high diagonal means regimes are persistent.

In [ ]:
prim.transition.round(3)

## E3. Chart: when does the model see stress?
**Send me this.** Top: the spread with marked events (red). Bottom: the model's filtered stress probability (orange).

In [ ]:
import matplotlib.pyplot as plt

lvl = weekly_level_bp(sp["baa_30y"])
plot_regimes(lvl, prim.stress_prob, "Baa - 30Y: filtered stress probability (orange) and marked events (red)", events=EVENTS)
plt.show()

## E4. Which events did the model catch?
**Send me this.** An empty `first_alert` means the model never flagged that event as stress. `lag_days`: gap between the alert and the event's start (event dates are approximate, the lag is indicative).

In [ ]:
detection_table(prim.stress_prob, EVENTS).round(2)

## E5. Does the model find stress episodes not on my list?
**Send me this.** Episodes lasting at least 2 weeks.

In [ ]:
regime_periods(prim.stress_prob, level_bp=lvl, min_weeks=2).round(2)

## E6. Robustness check: does Baa-10Y find the same regimes?
**Send me this.** High agreement = the result doesn't hinge on which proxy we picked.

In [ ]:
rob = fit_regime_model(df["baa10y"].dropna())
common = prim.stress_prob.index.intersection(rob.stress_prob.index)
agree = ((prim.stress_prob[common] >= 0.5) == (rob.stress_prob[common] >= 0.5)).mean() * 100
corr = prim.stress_prob[common].corr(rob.stress_prob[common])
print(f"Baa-30Y and Baa-10Y agree on the regime: {agree:.1f}% of weeks | probability correlation: {corr:.2f} | did the Baa-10Y model converge: {rob.converged}")
rob.summary.round(2)

## E7. Model selection: is 2 regimes actually better than 3?
So far we assumed 2 regimes (calm/stress) without justifying that choice. Here we fit a 3-regime version of the SAME model and compare by AIC and BIC. A 3-regime model has more parameters and can only fit the in-sample data at least as well, so a fair comparison must penalize that extra flexibility -- which is exactly what AIC/BIC do (BIC penalizes harder). If BIC does not improve when adding a third regime, that's evidence against a 3-state model for this series, not proof there are only two "true" states.
**Send me this.** This cell can take a minute or two (each k fits with several random restarts). Note also the standard errors already shown in E1's summary table (`weekly_sigma_se_pct`, `mean_se_pct`): the regime means are usually not statistically significant (their SE is close to the estimate itself), which is why the write-up leans on volatility/duration/time-share, not on the means.

In [ ]:
from credit_engine.spreads.regimes import compare_regime_counts

model_comparison = compare_regime_counts(sp["baa_30y"], k_range=(2, 3))
model_comparison

# SECTION F: Live simulation (expanding window)
The model in Section E was fit on the **full sample** (in-sample): its parameters were found while also seeing the future. That's fine for learning the model's structure, but it's an optimistic way to measure performance.

Here the model is refit periodically (default: every 13 weeks, about one quarter), **using only the data available up to that point each time**. Between refits, it filters forward with the last fitted (fixed) parameters. This mimics what a live system would actually have known.

**Important:** the optimizer (EM + quasi-Newton) can sometimes land in a statistically equivalent but different local optimum even on identical data (regime labels can swap). This is optimizer noise, not look-ahead leakage; it is documented and tested separately in the code.

## F1. Run the live simulation
**Send me this.** `n_refits`: how many times the model was refit. This cell can take a little while (it fits the model dozens of times).

In [ ]:
from credit_engine.spreads.backtest import expanding_refit, compare_live_vs_insample

live = expanding_refit(sp["baa_30y"], min_train_weeks=104, refit_every=13)
print("Total number of refits:", live.n_refits)
live.refit_log.tail(8)

## F2. Live vs full-sample (in-sample) comparison
**Send me this.** `agreement_pct`: how often the two approaches agree on the regime call (stress/calm). A lower number is expected and normal: the live system works with less information, and that gap is itself an interesting finding.

In [ ]:
cmp = compare_live_vs_insample(live.stress_prob, prim.stress_prob)
for k, v in cmp.items():
    print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")

## F3. Chart: live vs in-sample stress probability
**Send me this.** How much do they diverge, especially at the start of a crisis (when the model has less to go on)?

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(prim.stress_prob.index, prim.stress_prob.values, label="In-sample (optimistic)", color="tab:blue", alpha=0.7)
ax.plot(live.stress_prob.index, live.stress_prob.values, label="Live (expanding window)", color="tab:orange", alpha=0.8)
ax.set_ylim(0, 1)
ax.set_title("Baa-30Y: in-sample vs live-simulation stress probability")
ax.legend()
plt.tight_layout()
plt.show()

## F4. Sensitivity check: does the starting data window (min_train_weeks) matter?
How many weeks of data should the model demand before its first fit? Too little = noisier early estimates; too much = no signal at all for the first few years. We try a few values and see how the agreement with the in-sample fit changes.
**Send me this.** This cell can take ~2 minutes (it reruns the model from scratch for 3 different settings).

In [ ]:
from credit_engine.spreads.backtest import min_train_weeks_sensitivity

sens = min_train_weeks_sensitivity(sp["baa_30y"], prim.stress_prob, candidates=[104, 156, 208],
                                   refit_every=26, search_reps=5)
sens.round(2)

# SECTION G: Risk contribution (DV01=CS01, but the real-world effect differs)
Section B showed: for a plain bullet bond under a parallel shift, CS01 is mathematically equal to DV01. Here we complete that finding with real data: **the sensitivities are equal, but how much rates and credit spreads actually move is very different.**

**Important (rating-appropriate spread series):** each bond must use the ICE OAS series that matches its own rating (AAA-like -> aaa_oas, BBB-like -> bbb_oas, HY-like -> hy_oas). Feeding the same spread series to all three bonds would be wrong: since DV01=CS01, `spread_share_pct` then collapses to `spread_vol/(rate_vol+spread_vol)` and comes out IDENTICAL for all three, regardless of which bond it is. Volatility is also measured over the **last 3 years only** (`window_years=3`); otherwise, e.g., the full history of Treasury yields since 1962 (including the extreme-rate 1980s) would badly overstate today's risk.

**Important (real curve, not flat):** DV01 and CS01 are both computed against the REAL Treasury curve (2/5/10/20/30Y), not a single flat rate -- via `risk.cs01.dv01_curve`, consistent with how CS01 already works.

**Important (correlation):** `rate_risk_contribution` and `spread_risk_contribution` are MARGINAL pieces and do not simply add up to the true portfolio risk unless rate and spread moves are uncorrelated. In reality they usually move together (often negatively: rates fall while spreads widen in a flight-to-quality). The correct total is `total_risk_1sd = sqrt(DV01^2*sigma_r^2 + CS01^2*sigma_s^2 + 2*DV01*CS01*rho*sigma_r*sigma_s)`, shown alongside the naive (uncorrelated) sum for comparison.

## G1. Risk contribution table for the three example bonds
**Send me this.** The `dominant` column shows which factor (rate or credit) actually carries the risk. `rho` is the realized correlation between weekly rate and spread changes; compare `naive_sum_1sd_$` (ignores rho) with `total_risk_1sd_$` (accounts for it) -- if rho is negative, the total should be noticeably SMALLER than the naive sum.

In [ ]:
from credit_engine.portfolio.risk_contribution import (risk_contribution_report, report_table,
                                                       spread_series_for_rating, rating_vol_ratio)
from credit_engine.pricing.curves import linear_curve

TENORS = [2, 5, 10, 20, 30]
treasury_curve = linear_curve(TENORS, [df[f"dgs{t}"].dropna().iloc[-1] for t in TENORS])   # the REAL curve, not flat
ICE_BUCKETS = {"aaa_oas": df["aaa_oas"], "bbb_oas": df["bbb_oas"], "hy_oas": df["hy_oas"]}
BOND_RATINGS = {"AAA-like  5y": "AAA", "BBB-like 10y": "BBB", "HY-like   7y": "BB (HY)"}   # must match the labels in BONDS

reports = []
for label, bond, spr in BONDS:   # the example bonds from SECTION B (AAA/BBB/HY-like)
    spread_series = spread_series_for_rating(BOND_RATINGS[label], ICE_BUCKETS)
    reports.append(risk_contribution_report(label, bond, SETTLE, treasury_curve, spr, 10_000_000,
                                            rate_series=df["dgs10"], spread_series=spread_series))
report_table(reports).round(2)

# SECTION H: Stress scenarios
We test our example portfolio (AAA/BBB/HY-like, 10mm notional each) under a few scenarios using an **exact reprice** (not a duration approximation), on the same real Treasury curve as Section G: a parallel shock, a curve steepener, and historical-crisis-style spread widening.

## H1. Parallel shock and steepener scenarios
**Send me this.**

In [ ]:
from credit_engine.portfolio.scenarios import PortfolioBond, parallel, steepen, run_scenarios

portfolio = [PortfolioBond(label, bond, spr, 10_000_000) for label, bond, spr in BONDS]
basic_scenarios = [parallel(100), parallel(-100), steepen(-50, 100)]
run_scenarios(portfolio, SETTLE, treasury_curve, basic_scenarios).round(0)

## H2. Spread widening styled on historical crises, SCALED BY RATING
We take the real peak spread values from Section E/F's `regime_periods` output (measured on the long-history Baa-30Y proxy) and ask: what happens to our portfolio if spreads jump from today's level to that peak?

**Important:** applying the SAME bp widening to every bond regardless of rating would be unrealistic -- in a real crisis, HY spreads widen far more than AAA spreads. Each bond's widening is scaled by `rating_vol_ratio`: its own rating bucket's realized volatility relative to the Baa-30Y proxy the peaks were measured on.
**Send me this.**

In [ ]:
from credit_engine.portfolio.scenarios import scaled_historical_scenarios

current_bp = float(lvl.iloc[-1])   # today's Baa-30Y level (bp), from SECTION E
peaks_bp = {"2008 GFC": 620.0, "COVID-19": 430.0, "2022 rate shock": 340.0}   # approximate, based on E4/E5 observations
rating_ratios = {label: rating_vol_ratio(BOND_RATINGS[label], ICE_BUCKETS, sp["baa_30y"]) for label, _, _ in BONDS}
print("Rating volatility ratios vs Baa-30Y (i.e. how much more/less this bond's spread typically moves):", 
     {k: round(v, 2) for k, v in rating_ratios.items()})
hist_scenarios = scaled_historical_scenarios(peaks_bp, current_spread_bp=current_bp, rating_ratios=rating_ratios)
print(f"Today's Baa-30Y level: {current_bp:.0f} bp")
run_scenarios(portfolio, SETTLE, treasury_curve, hist_scenarios).round(0)

# SECTION I: Advanced interactive dashboard
Three tabs, built with `ipywidgets` (no separate web server needed), tying together everything built so far:
- **Pricing & Risk**: type in a bond, see its price, duration, DV01/CS01 and which risk factor (rate or credit) actually dominates, with a small chart.
- **Scenario Shock**: drag a curve shock and a spread shock, see the exact repriced P&L on that same bond update instantly.
- **Regime**: today's live (filtered) stress probability, plotted against the last few years, so you can see where "now" sits.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"], capture_output=True)

## I1. Dashboard
Run the cell, then switch between the three tabs and drag the sliders -- everything updates live.

In [ ]:
import ipywidgets as W
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, clear_output
from datetime import date, timedelta

from credit_engine.pricing.curves import shift_curve
from credit_engine.pricing.bond import dirty_price_from_curve

RATING_SPREAD_BP = {"AAA": 60, "AA": 90, "A": 130, "BBB": 180, "BB (HY)": 320, "B (HY)": 480, "CCC (HY)": 800}

# ---- shared bond inputs (drive all three tabs) ----
coupon_w = W.FloatSlider(value=6.0, min=0.0, max=12.0, step=0.25, description="Coupon (%)")
maturity_w = W.IntSlider(value=7, min=1, max=30, step=1, description="Maturity (yrs)")
rating_w = W.Dropdown(options=list(RATING_SPREAD_BP), value="BBB", description="Rating")
notional_w = W.FloatText(value=10_000_000, description="Notional ($)")

# ---- scenario-tab-only inputs ----
parallel_shock_w = W.FloatSlider(value=0, min=-300, max=300, step=5, description="Parallel shock (bp)")
spread_shock_w = W.FloatSlider(value=0, min=-300, max=300, step=5, description="Spread shock (bp)")

out_price, out_scenario, out_regime = W.Output(), W.Output(), W.Output()


def current_bond():
    maturity_date = SETTLE + timedelta(days=int(maturity_w.value * 365.25))
    bond = Bond(coupon_w.value / 100, maturity_date)
    spr = RATING_SPREAD_BP[rating_w.value] / 1e4
    spread_series = spread_series_for_rating(rating_w.value, ICE_BUCKETS)
    return bond, spr, spread_series


def render_pricing(_=None):
    with out_price:
        clear_output(wait=True)
        bond, spr, spread_series = current_bond()
        curve = treasury_curve
        r = risk_contribution_report(rating_w.value, bond, SETTLE, curve, spr, notional_w.value,
                                    rate_series=df["dgs10"], spread_series=spread_series)
        years_to_maturity = (bond.maturity - SETTLE).days / 365.25
        ytm = float(np.atleast_1d(curve(np.array([years_to_maturity])))[0]) + spr
        price = dirty_price_from_curve(bond, SETTLE, curve, spr)
        print(f"YTM (approx, at this bond's own maturity on the curve): {ytm:.2%}   Clean price: {price - bond.accrued_interest(SETTLE):.3f}")
        print(f"Macaulay: {macaulay_duration(bond, SETTLE, ytm):.2f}   Modified: {modified_duration(bond, SETTLE, ytm):.2f}   Convexity: {convexity(bond, SETTLE, ytm):.1f}")
        print(f"DV01: ${r.dv01:,.0f}   CS01: ${r.cs01:,.0f}  (essentially equal under a parallel shift)")
        print(f"Rate risk contribution:   ${r.rate_risk_contribution:,.0f}  (typical weekly move {r.rate_vol_bp:.1f} bp)")
        print(f"Credit risk contribution: ${r.spread_risk_contribution:,.0f}  (typical weekly move {r.spread_vol_bp:.1f} bp)")
        print(f"Correlation (rate vs spread weekly changes): {r.rho:+.2f}")
        print(f"Naive sum (ignores correlation): ${r.naive_sum_1sd:,.0f}   Total risk (correlation-adjusted): ${r.total_risk_1sd:,.0f}")
        print(f"Dominant risk factor: {r.dominant_factor.upper()}")

        fig, ax = plt.subplots(figsize=(5, 1.8))
        bars = ax.barh(["Rate", "Credit"], [r.rate_risk_contribution, r.spread_risk_contribution],
                       color=["tab:blue", "tab:red"])
        ax.bar_label(bars, fmt="${:,.0f}")
        ax.set_title("Risk contribution ($, typical week)")
        plt.tight_layout()
        plt.show()


def render_scenario(_=None):
    with out_scenario:
        clear_output(wait=True)
        bond, spr, _spread_series = current_bond()
        base_curve = treasury_curve
        base_price = dirty_price_from_curve(bond, SETTLE, base_curve, spr)
        base_value = base_price / 100 * notional_w.value

        shocked_curve = shift_curve(base_curve, parallel_shock_w.value / 1e4)
        shocked_spread = spr + spread_shock_w.value / 1e4
        shocked_price = dirty_price_from_curve(bond, SETTLE, shocked_curve, shocked_spread)
        shocked_value = shocked_price / 100 * notional_w.value

        pnl = shocked_value - base_value
        print(f"Base market value:    ${base_value:,.0f}")
        print(f"Shocked market value: ${shocked_value:,.0f}")
        print(f"P&L: ${pnl:,.0f}  ({pnl / base_value:+.1%})")
        print(f"\n(Exact reprice, not a linear DV01 estimate -- convexity is fully captured.)")


def render_regime(_=None):
    with out_regime:
        clear_output(wait=True)
        current = live.stress_prob.iloc[-1]
        print(f"Current Baa-30Y stress probability (live/filtered): {current:.0%}")
        print("STRESS" if current >= 0.5 else "CALM", "regime right now (>=50% threshold)")

        cutoff = live.stress_prob.index.max() - timedelta(days=1095)   # last 3 years
        recent = live.stress_prob[live.stress_prob.index >= cutoff]
        fig, ax = plt.subplots(figsize=(8, 2.5))
        ax.fill_between(recent.index, recent.values, color="tab:orange", alpha=0.6)
        ax.axhline(0.5, color="k", lw=0.8, ls="--")
        ax.plot(recent.index[-1], current, "o", color="black")
        ax.set_ylim(0, 1)
        ax.set_title("Live stress probability, last 3 years")
        plt.tight_layout()
        plt.show()


def render_all(_=None):
    render_pricing(); render_scenario(); render_regime()


for w in (coupon_w, maturity_w, rating_w, notional_w):
    w.observe(render_all, names="value")
for w in (parallel_shock_w, spread_shock_w):
    w.observe(render_scenario, names="value")

bond_inputs = W.VBox([coupon_w, maturity_w, rating_w, notional_w])
tab1 = W.VBox([bond_inputs, out_price])
tab2 = W.VBox([bond_inputs, parallel_shock_w, spread_shock_w, out_scenario])
tab3 = out_regime

tabs = W.Tab(children=[tab1, tab2, tab3])
tabs.set_title(0, "Pricing & Risk")
tabs.set_title(1, "Scenario Shock")
tabs.set_title(2, "Regime")
display(tabs)
render_all()

# SECTION J: Model risk and limitations
A single, consolidated place for every known limitation of this engine (scattered mentions above are gathered here). This is written the way a model-validation document would: what the model assumes, where it breaks, and what it should never be used for without further work.

**Pricing and risk (Sections B, G, H)**
- Fixed-rate, non-callable, bullet bonds only. No callable/putable/floating/amortizing structures, no CDS, no default/recovery modeling.
- The 30/360 day-count has a known small discrepancy vs QuantLib (up to ~8bp in yield) specifically when a coupon date falls on the last day of February.
- Risk contribution and scenarios use a single (Treasury) curve for discounting; no key-rate duration, no separate spread CURVE (OAS is treated as a single flat number per bond, not a term structure), no basis/liquidity/funding risk.
- H2's rating-scaled widening (`rating_vol_ratio`) is an approximation: it scales a long-history proxy's peak widening by a RECENT (3y) relative-volatility ratio, not by that rating's own historical peak (which ICE OAS, at only ~3 years of history, cannot supply for 2008 or COVID).

**Risk contribution correlation (Section G)**
- `rho` (rate/spread correlation) is estimated over the same 3-year window as the volatilities; it is a point estimate with no confidence interval and can itself be unstable across different windows -- treat `total_risk_1sd` as indicative, not as a precise VaR-grade number.

**Regime model (Sections E, F)**
- The regime MEANS are not reliably identified (their standard error is close to the estimate itself, shown explicitly in E1/E7); only volatility, expected duration and time-share should be trusted.
- Only 2 vs 3 regimes were compared (E7); GARCH, threshold-autoregressive, and other volatility models were not benchmarked against the Markov-switching approach.
- The proxy (Baa-30Y) is a broad, aggregate series: it is blind to sector-specific shocks (2015-16 energy, 2023 regional-bank stress showed zero signal, Section E4).
- The optimizer (EM + quasi-Newton) is not always bit-reproducible even with a fixed seed; it can land in a statistically equivalent but differently-labeled local optimum, or rarely fail with a numerical singularity (handled with retries, but this is optimizer noise, not a guarantee of the single best fit).
- The live simulation (Section F) refits on an expanding window with fixed periodicity and a somewhat arbitrary `min_train_weeks`; the sensitivity check (F4) suggests this choice matters less than expected, but only 3 settings were tried.

**What this engine does NOT do (by design, left for a possible second project)**
- No economic backtest: no Sharpe ratio, drawdown, turnover, transaction costs, or hit rate for any signal built on the regime probability. Detecting a regime accurately is not the same as trading it profitably.
- No calibration analysis (Brier score, reliability diagrams, out-of-sample log score) of the regime probabilities themselves.
- No position-sizing or meta-labeling framework turning the regime signal into an actual trade.

**Engineering**
- This notebook is a demo/exploration layer. The actual test suite (100+ tests covering pricing, risk, data, regimes, backtesting, scenarios and risk contribution) lives in the accompanying repository, not in this notebook -- if evaluating this work, please look at the repository's `tests/` directory and CI configuration, not just this notebook.
- No production concerns (logging, monitoring, scheduling, alerting) are implemented; this is a research/analysis tool, not a live trading system. If this were productionized, it would need, at minimum, scheduled data refreshes with staleness alerts, model retraining triggers, and a fallback when FRED/ICE data is late or unavailable.

# Done
Send me: **F4 (table), G1, H1, H2, I1 (a screenshot from each of the three tabs, after playing with the sliders)**. Screenshots are fine.